# 07. Key Findings Summary: Glacier Evolution, Streamflow Contribution, and Drought

Merged from `07_key_findings` and `agu_runthrough`. Runs on one basin set: the final basins from `02_final_basin_set.ipynb`, with results from `03_glacier_streamflow_fraction.ipynb` (run both first).

Key questions:
1. How are glaciers evolving 2000-2019 (MB timeseries for all glaciers, by region)?
2. Using MB timeseries and historic streamflow data, how much are glaciers contributing to streamflow (full timeseries, peak ablation, pre- vs. post-2010 megadrought impact)?
3. How is the relative importance of glaciers changing through time and with drought?

**What was merged and what was dropped**
- Kept from the old 07: everything in Sections 0 to 3, the Chow test, the peak-water check, the regression (3d) and the export for Alexis.
- Taken from the AGU notebook: the peak absolute melt month cell and the two extra Section 2 maps (3-panel and megadrought).
- Fractions for pre- vs post-2010 periods and the Q-decline vs melt-change split (Sections 2c and 3c) use the mean of yearly ratios, matching Sections 2a and 3a. The AGU notebook used the ratio of pooled sums, which understates drought-year spikes and overstates the Q-decline share.
- Removed because they only existed to compare basin sets or chase problems in the old sets: loading three basin sets, duplicate-basin detection, grouped/leaf/major maps, and the checks on the 9 flagged basins and the WA1 ERA5 February anomaly. Open checks that still apply are in the last section.

**All written findings (the text cells and the SUMMARY) come from the old 143-basin run. Do not cite them until they are rewritten from the new output.**

## Cell 0: Setup, paths, data loading

In [1]:
import os, pickle, calendar, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from matplotlib.cm import ScalarMappable
from matplotlib.colors import to_hex
from scipy import stats
from scipy.stats import linregress
from sklearn.linear_model import LinearRegression
warnings.filterwarnings('ignore')

# ── Basin set ───────────────────────────────────────────────────────────────
# One basin set: the final basins from 02_final_basin_set.ipynb (each glacier
# in one basin only). Results come from 03_glacier_streamflow_fraction.ipynb.
BASIN_SET = 'final_basin_set'

# ── Paths ─────────────────────────────────────────────────────────────────
BASE_PATH    = '/Users/milliespencer/v2_CR2_OGGM_project_11_2026'
OUTPUT_ROOT  = f'{BASE_PATH}/Output'
CAMELS_FULL  = f'{BASE_PATH}/Full_CAMELS_all_basins/CAMELS_CL_v202201'
RGI_PATH     = f'{BASE_PATH}/files_chile_OGGM_climate_comparison/RGI_BNA_Clusters.csv'
BOUNDS_PATH  = f'{CAMELS_FULL}/camels_cl_boundaries/camels_cl_boundaries.shp'

FINAL_DIR    = f'{OUTPUT_ROOT}/final_basin_set'          # from 02
OUT_DIR      = f'{OUTPUT_ROOT}/results_final_basins'     # from 03; figures and tables here too

LAG_DIR      = f'{OUTPUT_ROOT}/regression/{BASIN_SET}/lag_analysis'   # from 04
DROUGHT_DIR  = f'{OUT_DIR}/megadrought_analysis'
os.makedirs(DROUGHT_DIR, exist_ok=True)
os.makedirs(LAG_DIR, exist_ok=True)

# ── Constants ────────────────────────────────────────────────────────────
YEARS         = list(range(2000, 2020))
MONTHS        = list(range(1, 13))
WATER_DENSITY = 1000
CLUSTERS      = ['OT3','DA1','DA2','DA3','WA1','WA2','WA3','WA4','WA5','WA6']
DATASETS      = {'CR2MET': 'TC', 'ERA5': 'ERA5', 'CRU': 'CRU'}
DS_COLORS     = {'CR2MET': '#1f77b4', 'ERA5': '#ff7f0e', 'CRU': '#2ca02c'}
REGION_LABELS = {
    'OT3': 'Altiplano',     'DA1': 'Dry Andes N',
    'DA2': 'Dry Andes C',   'DA3': 'Dry Andes S',
    'WA1': 'Wet Andes N',   'WA2': 'Wet Andes C',
    'WA3': 'N Patagonia',   'WA4': 'S Patagonia',
    'WA5': 'Patagonia',     'WA6': 'Tierra del Fuego',
}
BASELINE     = list(range(2000, 2010))
DROUGHT_DEFS = {
    'Baseline (2000-2009)':  list(range(2000, 2010)),
    'Drought (2010-2019)':   list(range(2010, 2020)),
}
DROUGHT_COLORS = {
    'Baseline (2000-2009)':  '#2ecc71',
    'Drought (2010-2019)':   '#f39c12',
}
SUMMER, WINTER = [12, 1, 2], [6, 7, 8]
FMA_MONTHS = [2, 3, 4]
PCTILE_THRESHOLD = 20   # basin-relative hydrologic drought threshold

def nc_path(dataset_name, cluster):
    tag = DATASETS[dataset_name]
    return (f'{OUTPUT_ROOT}/{dataset_name}/{cluster}/'
            f'run_output_2000_2019_hydro_{tag}_{cluster}.nc')

def sig_stars(p):
    """Standard significance stars. Used in EVERY stats cell in this
    notebook per project convention: always report p-value + stars."""
    if pd.isna(p):
        return 'n/a'
    if p < 0.001:
        return '***'
    if p < 0.01:
        return '**'
    if p < 0.05:
        return '*'
    return 'ns'

def basins_in_cluster(cluster):
    """Basins whose glaciers include this cluster (info['clusters'] is a list,
    main zone first). Use this instead of info['clusters'][0] filtering, which
    silently mishandles basins with glaciers from more than one cluster."""
    return [gid for gid, info in GAUGE_INFO_FRAC.items()
            if cluster in info['clusters']]

# ── Load RESULTS and the final basin set ────────────────────────────────────
print(f'BASIN_SET = "{BASIN_SET}"  |  loading from {OUT_DIR}')

with open(f'{OUT_DIR}/RESULTS.pkl', 'rb') as f:
    RESULTS = pickle.load(f)
print(f'  RESULTS loaded: {len(RESULTS["CR2MET"])} gauges')

assign      = pd.read_csv(f'{FINAL_DIR}/glacier_assignment.csv', dtype={'gauge_id': str})
final_table = pd.read_csv(f'{FINAL_DIR}/final_basin_table.csv', dtype={'gauge_id': str})
rgi_all_df  = pd.read_csv(RGI_PATH)

rgi_ids_by_gauge = assign.groupby('gauge_id')['RGIId'].apply(list).to_dict()
cen_lon_by_gauge = (assign[['RGIId', 'gauge_id']]
                    .merge(rgi_all_df[['RGIId', 'CenLon']], on='RGIId')
                    .groupby('gauge_id')['CenLon'].mean().to_dict())

GAUGE_INFO_FRAC = {}
for gauge_id, res in RESULTS['CR2MET'].items():
    if res is None or gauge_id not in set(final_table['gauge_id']):
        continue
    GAUGE_INFO_FRAC[gauge_id] = {
        'gauge_name':       res['gauge_name'],
        'rgi_ids':          rgi_ids_by_gauge.get(gauge_id, []),
        'n_glaciers':       res['n_glaciers'],
        'glacier_area_km2': res['glacier_area_km2'],
        'basin_area_km2':   res['basin_area_km2'],
        'clusters':         res['clusters'],             # main zone first
        'cen_lat':          res['cen_lat'],
        'cen_lon':          cen_lon_by_gauge.get(gauge_id, np.nan),
        'complete_years':   res['complete_years'],
    }

# Basins dropped by the high-fraction check in notebook 03 (mean annual fraction > 100%)
_excl = f'{OUT_DIR}/excluded_high_frac_gauges.csv'
if os.path.exists(_excl):
    excluded = pd.read_csv(_excl, dtype={'gauge_id': str})
    for gauge_id in excluded['gauge_id'].tolist():
        GAUGE_INFO_FRAC.pop(gauge_id, None)
    print(f'  Dropped by the high-fraction check: {len(excluded)}')

print(f'  GAUGE_INFO_FRAC: {len(GAUGE_INFO_FRAC)} basins')
print('Cell 0 complete')

BASIN_SET = "final_basin_set"  |  loading from /Users/milliespencer/v2_CR2_OGGM_project_11_2026/Output/results_final_basins


FileNotFoundError: [Errno 2] No such file or directory: '/Users/milliespencer/v2_CR2_OGGM_project_11_2026/Output/results_final_basins/RESULTS.pkl'

## Start with model validation and dataset comparison

In [ ]:
from oggm import utils
geodetic_ref = utils.get_geodetic_mb_dataframe()
geodetic_ref = geodetic_ref[geodetic_ref['period'] == '2000-01-01_2020-01-01']

base_path = f'{OUTPUT_ROOT}/'
datasets  = ['CR2MET', 'ERA5', 'CRU']
colors    = {'CR2MET': '#e74c3c', 'ERA5': '#3498db', 'CRU': '#2ecc71'}

# glacier_lists built directly from RGI_BNA_Clusters.csv (the cluster's
# full glacier assignment), NOT from CAMELS gauge/basin rgi_ids -- the
# latter only returns glaciers attached to a streamflow gauge, which
# would silently drop WA5 (near-zero CAMELS gauges) despite it having
# 711 real, geodetically-matched glaciers.
rgi_clusters_df = pd.read_csv(RGI_PATH)

glacier_lists = {}
for cl in CLUSTERS:
    cl_rgi_ids = set(rgi_clusters_df[rgi_clusters_df['Cluster'] == cl]['RGIId'])
    glacier_lists[cl] = {
        'with_gmb': [g for g in cl_rgi_ids if geodetic_ref is not None
                     and g in geodetic_ref.index]
    }
    print(f'{cl}: {len(glacier_lists[cl]["with_gmb"])} glaciers with GMB')

In [ ]:
print('N glaciers and total area (km2) per dataset per cluster:')
header = f'{"Cluster":<8}' + ''.join(f'{ds:>10}{"(km2)":>8}' for ds in datasets + ['GMB [m w.e./yr]'])
print(header)
print('-' * (8 + 18 * (len(datasets) + 1)))

for cl in CLUSTERS:
    row = f'{cl:<8}'
    for dataset in datasets:
        mb_file = f'{base_path}{dataset}/{cl}/mb.csv'
        if os.path.exists(mb_file) and geodetic_ref is not None:
            mb_df     = pd.read_csv(mb_file).set_index('rgi_id')
            year_cols = mb_df.columns[:20]
            ran       = mb_df[year_cols].dropna(how='all').index.tolist()
            matched   = [g for g in ran if g in geodetic_ref.index]
            n, area   = len(matched), geodetic_ref.loc[matched, 'area'].sum() / 1e6
        else:
            n, area = 0, 0.0
        row += f'{n:>10}{area:>7.1f} '
    if geodetic_ref is not None:
        gmb_glaciers = glacier_lists[cl]['with_gmb']
        n_gmb    = len(gmb_glaciers)
        area_gmb = geodetic_ref.loc[gmb_glaciers, 'area'].sum() / 1e6
        row += f'{n_gmb:>10}{area_gmb:>7.1f} '
    print(row)

print('\nSection 0 complete -- proceed to Section 1 (Q1: MB evolution)')

In [ ]:
# SMB vs GMB validation, per-glacier boxplot, ALL clusters (shared y-axis, tall panels)

CLUSTER_DATA = {}
global_min, global_max = np.inf, -np.inf

for cl in CLUSTERS:
    cl_glacier_data = {}
    for dataset in datasets:
        mb_file = f'{base_path}{dataset}/{cl}/mb.csv'
        if not os.path.exists(mb_file):
            continue
        mb_df = pd.read_csv(mb_file).set_index('rgi_id')
        year_cols = mb_df.columns[:20]
        cl_glacier_data[dataset] = (mb_df[year_cols].mean(axis=1) / 1000).dropna()

    cl_matched = glacier_lists[cl]['with_gmb']
    cl_gmb, cl_gmb_err = np.nan, np.nan
    if cl_matched and geodetic_ref is not None:
        cl_weights = geodetic_ref.loc[cl_matched, 'area']
        cl_gmb = np.average(geodetic_ref.loc[cl_matched, 'dmdtda'] * 1000, weights=cl_weights) / 1000
        cl_gmb_err = np.average(geodetic_ref.loc[cl_matched, 'err_dmdtda'] * 1000, weights=cl_weights) / 1000

    CLUSTER_DATA[cl] = {'glacier_data': cl_glacier_data, 'gmb': cl_gmb, 'gmb_err': cl_gmb_err}

    for vals in cl_glacier_data.values():
        if len(vals) > 0:
            global_min = min(global_min, vals.min())
            global_max = max(global_max, vals.max())
    if not np.isnan(cl_gmb):
        global_min = min(global_min, cl_gmb - cl_gmb_err)
        global_max = max(global_max, cl_gmb + cl_gmb_err)

y_range = global_max - global_min
YLIM = (global_min - 0.05*y_range, global_max + 0.05*y_range)

fig, axes = plt.subplots(5, 2, figsize=(14, 30))
axes_flat = axes.flatten()

for ax, cl in zip(axes_flat, CLUSTERS):
    cl_glacier_data = CLUSTER_DATA[cl]['glacier_data']
    cl_gmb, cl_gmb_err = CLUSTER_DATA[cl]['gmb'], CLUSTER_DATA[cl]['gmb_err']

    box_data   = [cl_glacier_data[ds].values for ds in datasets if ds in cl_glacier_data]
    box_labels = [ds for ds in datasets if ds in cl_glacier_data]
    if not box_data:
        ax.set_visible(False)
        continue

    bp = ax.boxplot(box_data, labels=box_labels, patch_artist=True, showmeans=True, widths=0.6,
                    meanprops=dict(marker='D', markerfacecolor='red', markersize=8, markeredgecolor='black'))
    for patch, ds in zip(bp['boxes'], box_labels):
        patch.set_facecolor(colors[ds])
        patch.set_alpha(0.7)

    if not np.isnan(cl_gmb):
        ax.axhline(cl_gmb, color='darkred', ls='--', lw=1.8, label=f'GMB: {cl_gmb:.3f} +/- {cl_gmb_err:.3f}')
        ax.axhspan(cl_gmb-cl_gmb_err, cl_gmb+cl_gmb_err, color='gray', alpha=0.12)
        ax.legend(fontsize=9, loc='best')

    ax.set_title(f'{cl} -- {REGION_LABELS[cl]}', fontsize=13, fontweight='bold')
    ax.set_ylim(YLIM)
    ax.set_ylabel('MB (m w.e./yr)', fontsize=11)
    ax.tick_params(labelsize=10)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Section 0.2: Per-Glacier MB vs Hugonnet GMB, All Clusters (2000-2019)\n'
             'Dashed line = GMB, shaded = GMB uncertainty -- validates calibration\n'
             f'(shared y-axis: {YLIM[0]:.2f} to {YLIM[1]:.2f} m w.e./yr)',
             fontsize=15, fontweight='bold', y=1.005)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/section0_2_mb_validation_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Dataset spread: national runoff decomposition (melt/snowfall/liq_prcp)
# Tests whether inter-dataset disagreement is PRECIPITATION-driven or
# TEMPERATURE-driven, exploiting OGGM's mu_star calibration constraint:
# every dataset's simulated MB is forced to match the same Hugonnet
# geodetic MB target, so if one dataset simulates more snowfall, melt
# MUST also be higher to still hit that same target MB.

results_spread = []
for ds_name, tag in DATASETS.items():
    total_melt = total_snowfall = total_liq_prcp = 0.0
    for cl in CLUSTERS:
        f = nc_path(ds_name, cl)
        if not os.path.exists(f):
            continue
        ds = xr.open_dataset(f)
        melt     = ds['melt_on_glacier_monthly'].values[1:, :, :]
        snowfall = ds['snowfall_on_glacier_monthly'].values[1:, :, :]
        liq_prcp = ds['liq_prcp_on_glacier_monthly'].values[1:, :, :]
        ds.close()
        total_melt     += np.nansum(melt)
        total_snowfall += np.nansum(snowfall)
        total_liq_prcp += np.nansum(liq_prcp)

    n = len(YEARS)
    results_spread.append({
        'dataset': ds_name, 'melt_km3yr': total_melt/1e12/n,
        'snowfall_km3yr': total_snowfall/1e12/n, 'liq_prcp_km3yr': total_liq_prcp/1e12/n,
    })

df_spread = pd.DataFrame(results_spread).set_index('dataset')

print('Section 0.3: National glacier water fluxes by dataset')
print('UNITS: km3 water-equiv per year (averaged 2000-2019, n=19 yrs)')
print('='*75)
print(f'{"Dataset":8s} {"Melt":>10s} {"Snowfall":>10s} {"Liq.prcp":>10s}')
print('-'*75)
for ds_name, row in df_spread.iterrows():
    print(f'{ds_name:8s} {row["melt_km3yr"]:10.1f} {row["snowfall_km3yr"]:10.1f} {row["liq_prcp_km3yr"]:10.1f}')

fig, axes = plt.subplots(1, 3, figsize=(16, 8))
spread_stats = {}
for ax, col, title in zip(axes, ['melt_km3yr', 'snowfall_km3yr', 'liq_prcp_km3yr'],
    ['Melt on glacier\n(ice melt + snowmelt)', 'Snowfall on glacier\n(solid precip input)',
     'Liquid precip on glacier\n(rain, runs off immediately)']):
    vals   = df_spread[col]
    spread = vals.max() - vals.min()
    ratio  = vals.max() / vals.min()
    spread_stats[col] = spread

    bars = ax.bar(vals.index, vals.values, color=[DS_COLORS[d] for d in vals.index], edgecolor='k', lw=0.5)
    ax.set_title(title, fontsize=11)
    ax.set_ylabel('km3 water-equivalent per year', fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(0, vals.max() * 1.35)

    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + vals.max()*0.02, f'{val:.1f} km3/yr', ha='center', fontsize=10)

    ax.text(0.5, 0.97, f'Absolute disagreement (max-min): {spread:.1f} km3/yr\n'
            f'Relative disagreement (max/min): {ratio:.2f}x ({vals.idxmax()} vs {vals.idxmin()})',
            transform=ax.transAxes, ha='center', va='top', fontsize=8.5,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

fig.suptitle('Section 0.3: How Much Do CR2MET, ERA5, and CRU Disagree, and Why?\n'
             'National totals, all Chilean glaciers, 2000-2019 average (km3 water-equiv./yr)',
             fontsize=14, y=1.03)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/section0_3_dataset_spread_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()

melt_spread = spread_stats['melt_km3yr']
snow_spread = spread_stats['snowfall_km3yr']
diff_pct = abs(melt_spread - snow_spread) / max(melt_spread, snow_spread) * 100

print(f'\n{"="*75}')
print('INTERPRETATION: Why do the three datasets disagree?')
print(f'{"="*75}')
print(f'Melt disagreement (max-min):     {melt_spread:.1f} km3/yr')
print(f'Snowfall disagreement (max-min): {snow_spread:.1f} km3/yr')
print(f'These two numbers differ by only {diff_pct:.1f}%.\n')

if diff_pct < 20:
    print('CONCLUSION: Melt disagreement approx. equals snowfall disagreement.')
    print('The three datasets mainly disagree about HOW MUCH SNOW FALLS on the')
    print('glaciers (precipitation), not about melt efficiency (temperature).')
    print('Because OGGM calibrates every dataset to hit the SAME real-world MB')
    print('target (Hugonnet geodetic data), a dataset with more snow is forced')
    print('to also produce more melt just to keep MB correct -- melt differences')
    print('are mostly a downstream CONSEQUENCE of precipitation differences.')
else:
    print('CONCLUSION: Melt disagreement differs substantially from snowfall')
    print('disagreement -- datasets likely disagree about melt for reasons beyond')
    print('precipitation input alone.')

# SECTION 1 -- Q1: How are glaciers evolving from 2000-2019?

In [ ]:
# Annual area-weighted MB timeseries, all clusters, all datasets (shared y-axis)

MB_ANNUAL = {}
CLUSTER_GMB = {}
CLUSTERS_WITH_DATA = []

for cl in CLUSTERS:
    MB_ANNUAL[cl] = {}
    cl_matched_all = glacier_lists[cl]['with_gmb']
    if not cl_matched_all or geodetic_ref is None:
        continue

    cl_weights = geodetic_ref.loc[cl_matched_all, 'area']
    cl_gmb = np.average(geodetic_ref.loc[cl_matched_all, 'dmdtda'] * 1000, weights=cl_weights) / 1000
    cl_gmb_err = np.average(geodetic_ref.loc[cl_matched_all, 'err_dmdtda'] * 1000, weights=cl_weights) / 1000
    CLUSTER_GMB[cl] = (cl_gmb, cl_gmb_err)

    for dataset in datasets:
        mb_file = f'{base_path}{dataset}/{cl}/mb.csv'
        if not os.path.exists(mb_file):
            continue
        mb_df = pd.read_csv(mb_file).set_index('rgi_id')
        year_cols = mb_df.columns[:20]
        matched = [g for g in cl_matched_all if g in mb_df.index]
        if not matched:
            continue
        weights = geodetic_ref.loc[matched, 'area'].values
        annual_mb = [np.average(mb_df.loc[matched, c].values, weights=weights) for c in year_cols]
        ts = pd.DataFrame({'year': range(2000, 2020), 'mb': np.array(annual_mb) / 1000})
        MB_ANNUAL[cl][dataset] = ts

    if MB_ANNUAL[cl]:
        CLUSTERS_WITH_DATA.append(cl)

excluded_clusters = [cl for cl in CLUSTERS if cl not in CLUSTERS_WITH_DATA]
if excluded_clusters:
    print(f'Clusters excluded (no MB data): {excluded_clusters}')

global_min, global_max = np.inf, -np.inf
for cl in CLUSTERS_WITH_DATA:
    for ts in MB_ANNUAL[cl].values():
        global_min = min(global_min, ts['mb'].min())
        global_max = max(global_max, ts['mb'].max())
    cl_gmb, cl_gmb_err = CLUSTER_GMB[cl]
    global_min = min(global_min, cl_gmb - cl_gmb_err)
    global_max = max(global_max, cl_gmb + cl_gmb_err)

y_range = global_max - global_min
YLIM = (global_min - 0.05*y_range, global_max + 0.05*y_range)

n_clusters = len(CLUSTERS_WITH_DATA)
ncols = 3
nrows = int(np.ceil(n_clusters / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(18, 5.5*nrows), sharex=True)
axes_flat = axes.flatten()

for ax, cl in zip(axes_flat, CLUSTERS_WITH_DATA):
    cl_gmb, cl_gmb_err = CLUSTER_GMB[cl]
    for dataset in datasets:
        if dataset not in MB_ANNUAL[cl]:
            continue
        ts = MB_ANNUAL[cl][dataset]
        mb_file = f'{base_path}{dataset}/{cl}/mb.csv'
        mb_df = pd.read_csv(mb_file).set_index('rgi_id')
        matched = [g for g in glacier_lists[cl]['with_gmb'] if g in mb_df.index]
        ax.plot(ts['year'], ts['mb'], 'o-', color=colors[dataset], lw=1.5, ms=4,
                label=f'{dataset} (N={len(matched)})', alpha=0.85)

    ax.axhline(cl_gmb, color='darkred', ls=':', lw=1.8, label=f'GMB: {cl_gmb:.3f}+/-{cl_gmb_err:.3f}')
    ax.axhspan(cl_gmb-cl_gmb_err, cl_gmb+cl_gmb_err, color='gray', alpha=0.1)
    ax.axhline(0, color='k', lw=0.6)
    ax.set_ylim(YLIM)
    ax.set_title(f'{cl} -- {REGION_LABELS[cl]}', fontsize=11, fontweight='bold')
    ax.tick_params(labelsize=8, axis='x', rotation=45)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7, loc='best')

for ax in axes_flat[n_clusters:]:
    ax.set_visible(False)
for i in range(0, n_clusters, ncols):
    axes_flat[i].set_ylabel('MB (m w.e./yr)', fontsize=10)

fig.suptitle('Section 1.1: Annual Area-Weighted Mass Balance by Cluster (2000-2019)\n'
             f'Shared y-axis: {YLIM[0]:.2f} to {YLIM[1]:.2f} m w.e./yr'
             + (f'  |  Excluded: {", ".join(excluded_clusters)} (no data)' if excluded_clusters else ''),
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/section1_1_mb_timeseries_all_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

mb_summary_records = []
for cl in CLUSTERS_WITH_DATA:
    for ds_name in datasets:
        if ds_name not in MB_ANNUAL[cl]:
            continue
        ts = MB_ANNUAL[cl][ds_name]
        base_vals = ts[ts['year'].isin(BASELINE)]['mb']
        dr_ts     = ts[ts['year'].isin(range(2010,2020))]
        dr_vals   = dr_ts['mb']
        base_mb, dr_mb = base_vals.mean(), dr_vals.mean()

        t_stat, p_period = stats.ttest_ind(dr_vals, base_vals)
        slope_full, _, _, p_trend_full, _ = linregress(ts['year'], ts['mb'])
        if len(dr_ts) >= 4:
            slope_post, _, _, p_trend_post, _ = linregress(dr_ts['year'], dr_ts['mb'])
        else:
            slope_post, p_trend_post = np.nan, np.nan

        mb_summary_records.append({
            'cluster': cl, 'dataset': ds_name, 'mb_2000_2009': base_mb, 'mb_2010_2019': dr_mb,
            'mb_change': dr_mb - base_mb, 'period_p': p_period, 'period_sig': sig_stars(p_period),
            'trend_full_slope': slope_full, 'trend_full_p': p_trend_full, 'trend_full_sig': sig_stars(p_trend_full),
            'trend_post2010_slope': slope_post, 'trend_post2010_p': p_trend_post, 'trend_post2010_sig': sig_stars(p_trend_post),
        })

df_mb_summary = pd.DataFrame(mb_summary_records)
df_mb_summary.to_csv(f'{OUT_DIR}/section1_2_mb_summary_stats.csv', index=False)

print('Section 1.2: MB trend significance -- full record vs. post-2010-only')
print(f'{"Cluster":6s} {"Dataset":8s} {"Full trend p":13s} {"sig":4s}  {"Post-2010 trend p":18s} {"sig"}')
print('-'*100)
for _, row in df_mb_summary.iterrows():
    print(f'{row["cluster"]:6s} {row["dataset"]:8s} {row["trend_full_p"]:13.4f} {row["trend_full_sig"]:4s}  '
          f'{row["trend_post2010_p"]:18.4f} {row["trend_post2010_sig"]}')

print()
print('Section 1.2b: Period comparison (t-test), 2000-2009 vs 2010-2019 means')
print(f'{"Cluster":6s} {"Dataset":8s} {"2000-09":9s} {"2010-19":9s} {"Change":8s} {"period_p":9s} {"sig"}')
print('-'*80)
for _, row in df_mb_summary.iterrows():
    print(f'{row["cluster"]:6s} {row["dataset"]:8s} {row["mb_2000_2009"]:9.3f} {row["mb_2010_2019"]:9.3f} '
          f'{row["mb_change"]:+8.3f} {row["period_p"]:9.4f} {row["period_sig"]}')

In [ ]:
# Section 1.2c: Is ERA5 "noisier" or genuinely showing a different signal?
print('MB series variability by cluster and dataset (SD of annual MB, m w.e./yr)')
print('='*90)
print(f'{"Cluster":6s} {"CR2MET SD":10s} {"ERA5 SD":10s} {"CRU SD":10s}  '
      f'{"CR2MET slope":13s} {"ERA5 slope":11s} {"CRU slope":10s}')
print('-'*90)

variability_records = []
for cl in CLUSTERS_WITH_DATA:
    sds, slopes = {}, {}
    for ds_name in datasets:
        if ds_name not in MB_ANNUAL[cl]:
            continue
        ts = MB_ANNUAL[cl][ds_name]
        sds[ds_name] = ts['mb'].std()
        slope, _, _, _, _ = linregress(ts['year'], ts['mb'])
        slopes[ds_name] = slope

    variability_records.append({'cluster': cl, **{f'{d}_sd': sds.get(d, np.nan) for d in datasets},
                                **{f'{d}_slope': slopes.get(d, np.nan) for d in datasets}})

    print(f'{cl:6s} {sds.get("CR2MET",np.nan):10.3f} {sds.get("ERA5",np.nan):10.3f} '
          f'{sds.get("CRU",np.nan):10.3f}  {slopes.get("CR2MET",np.nan):13.4f} '
          f'{slopes.get("ERA5",np.nan):11.4f} {slopes.get("CRU",np.nan):10.4f}')

df_variability = pd.DataFrame(variability_records)
df_variability.to_csv(f'{OUT_DIR}/section1_2c_variability_check.csv', index=False)

print()
print('INTERPRETATION GUIDE:')
print('If ERA5_sd > CR2MET_sd and ERA5_sd > CRU_sd, AND ERA5_slope is similar in')
print('  sign/magnitude to the other two -> ERA5 is noisier but not wrong (a power issue).')
print('If ERA5_slope is smaller in magnitude or flips sign vs. the other two')
print('  -> ERA5 genuinely disagrees about the underlying climate signal.')

n_era5_noisiest = sum(1 for _, row in df_variability.iterrows()
                      if row.get('ERA5_sd', 0) > row.get('CR2MET_sd', np.inf)
                      and row.get('ERA5_sd', 0) > row.get('CRU_sd', np.inf))
print(f'\\nERA5 is the noisiest of the 3 in {n_era5_noisiest}/{len(df_variability)} clusters.')

In [ ]:
# Section 1.2d: Chow test -- formal test for a structural break at 2010
# Tests whether MB is better described by ONE continuous trend across
# 2000-2019, or by TWO separate regimes (pre-2010, post-2010) with a
# break in between. This is the standard test for exactly the "step-
# change vs. continuous decline" question -- not just eyeballing two
# separate p-values.

def chow_test(years, mb, break_year=2010):
    years = np.asarray(years, dtype=float)
    mb = np.asarray(mb, dtype=float)
    pre_mask  = years < break_year
    post_mask = years >= break_year
    n1, n2 = pre_mask.sum(), post_mask.sum()
    if n1 < 3 or n2 < 3:
        return np.nan, np.nan, np.nan

    def rss_of_fit(x, y):
        slope, intercept, r, p, se = linregress(x, y)
        resid = y - (slope * x + intercept)
        return np.sum(resid**2)

    rss_pooled = rss_of_fit(years, mb)
    rss_1 = rss_of_fit(years[pre_mask], mb[pre_mask])
    rss_2 = rss_of_fit(years[post_mask], mb[post_mask])
    rss_sep = rss_1 + rss_2

    k = 2   # params per line: intercept + slope
    n = n1 + n2
    numerator = (rss_pooled - rss_sep) / k
    denominator = rss_sep / (n - 2*k)
    if denominator <= 0:
        return np.nan, np.nan, np.nan

    F = numerator / denominator
    p_val = stats.f.sf(F, k, n - 2*k)
    return F, p_val, sig_stars(p_val)

print('Section 1.2d: Chow test for structural break at 2010')
print('(Tests whether pre/post-2010 regimes differ significantly from a single continuous trend)')
print('='*90)
print(f'{"Cluster":6s} {"Dataset":8s} {"Chow F":10s} {"Chow p":10s} {"sig":4s}  {"Interpretation"}')
print('-'*90)

chow_records = []
for cl in CLUSTERS_WITH_DATA:
    for ds_name in datasets:
        if ds_name not in MB_ANNUAL[cl]:
            continue
        ts = MB_ANNUAL[cl][ds_name]
        F, p, sig = chow_test(ts['year'].values, ts['mb'].values, break_year=2010)
        if np.isnan(F):
            continue
        interp = 'STRUCTURAL BREAK (step-change)' if p < 0.05 else 'no significant break (single trend adequate)'
        chow_records.append({'cluster': cl, 'dataset': ds_name, 'chow_F': F, 'chow_p': p, 'chow_sig': sig})
        print(f'{cl:6s} {ds_name:8s} {F:10.3f} {p:10.4f} {sig:4s}  {interp}')

df_chow = pd.DataFrame(chow_records)
df_chow.to_csv(f'{OUT_DIR}/section1_2d_chow_test.csv', index=False)

n_sig = df_chow['chow_p'].lt(0.05).sum()
print()
print(f'{n_sig}/{len(df_chow)} cluster-dataset combinations show a significant structural break at 2010.')

## no breakpoint, this is a monotonic trend 

In [ ]:
# Glacier volume evolution, all clusters, all datasets
def load_cluster_totals(cluster, dataset):
    f = nc_path(dataset, cluster)
    if not os.path.exists(f):
        return None
    ds = xr.open_dataset(f)
    vol = ds['volume'].sum(dim='rgi_id').values / 1e9
    years = ds['time'].values
    ds.close()
    return pd.DataFrame({'year': years, 'volume_km3': vol})

fig, axes = plt.subplots(2, 5, figsize=(22, 9), sharex=True)
axes_flat = axes.flatten()
VOLUME_TOTALS = {}

for ax, cl in zip(axes_flat, CLUSTERS):
    VOLUME_TOTALS[cl] = {}
    for ds_name in datasets:
        ts = load_cluster_totals(cl, ds_name)
        if ts is None:
            continue
        ts = ts[ts['year'].isin(YEARS)]
        VOLUME_TOTALS[cl][ds_name] = ts
        ax.plot(ts['year'], ts['volume_km3'], 'o-', color=DS_COLORS[ds_name], lw=1.5, ms=3, label=ds_name, alpha=0.85)
    ax.set_title(f'{cl} -- {REGION_LABELS[cl]}', fontsize=9, fontweight='bold')
    ax.tick_params(labelsize=7, axis='x', rotation=45)
    ax.grid(True, alpha=0.3)

axes_flat[0].legend(fontsize=7, loc='best')
axes_flat[0].set_ylabel('Volume (km3)', fontsize=9)
axes_flat[5].set_ylabel('Volume (km3)', fontsize=9)
fig.suptitle('Section 1.3: Glacier Volume Evolution by Cluster (2000-2019)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/section1_3_volume_evolution_all_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

vol_summary_records = []
for cl in CLUSTERS:
    for ds_name in datasets:
        if cl not in VOLUME_TOTALS or ds_name not in VOLUME_TOTALS[cl]:
            continue
        ts = VOLUME_TOTALS[cl][ds_name]
        if len(ts) < 5:
            continue
        v0, v1 = ts.iloc[0]['volume_km3'], ts.iloc[-1]['volume_km3']
        pct_change = (v1 - v0) / v0 * 100 if v0 != 0 else np.nan
        slope, intercept, r, p, se = linregress(ts['year'], ts['volume_km3'])
        vol_summary_records.append({
            'cluster': cl, 'dataset': ds_name, 'volume_2000_km3': v0, 'volume_2019_km3': v1,
            'pct_change': pct_change, 'trend_km3_per_yr': slope, 'trend_p': p, 'trend_sig': sig_stars(p),
        })

df_vol_summary = pd.DataFrame(vol_summary_records)
df_vol_summary.to_csv(f'{OUT_DIR}/section1_4_volume_summary_stats.csv', index=False)

print('Section 1.4: Volume change summary by cluster and dataset')
print(f'{"Cluster":6s} {"Dataset":8s} {"Vol 2000":9s} {"Vol 2019":9s} {"% chg":8s} {"Trend km3/yr":13s} {"p":8s} {"sig"}')
print('-'*95)
for _, row in df_vol_summary.iterrows():
    print(f'{row["cluster"]:6s} {row["dataset"]:8s} {row["volume_2000_km3"]:9.2f} {row["volume_2019_km3"]:9.2f} '
          f'{row["pct_change"]:+7.1f}% {row["trend_km3_per_yr"]:13.4f} {row["trend_p"]:8.4f} {row["trend_sig"]}')

Glacier volume declined significantly in every cluster and every climate
dataset over 2000-2019, with relative losses ranging from ~1.4-1.9% in
WA5 to 22.4-23.0% in WA2. Unlike annual mass balance (sensitive to
climate-forcing choice in several clusters), cumulative volume loss
showed strong agreement across CR2MET, ERA5, and CRU in both magnitude
and direction -- the relative rate of glacier volume decline is robust,
largely independent of climate dataset choice.

In [ ]:
# Section 1.4: Accumulation vs. ablation decomposition (why is MB changing?)
accum_ablat_records = []
for ds_name in datasets:
    for cl in CLUSTERS:
        f = nc_path(ds_name, cl)
        if not os.path.exists(f):
            continue
        ds = xr.open_dataset(f)
        rgi_all = list(ds.rgi_id.values)
        actual_years = ds['time'].values.astype(int).tolist()

        snow_annual = np.nansum(ds['snowfall_on_glacier_monthly'].values, axis=1) / WATER_DENSITY
        melt_annual = np.nansum(ds['melt_on_glacier_monthly'].values, axis=1) / WATER_DENSITY
        ds.close()

        for gauge_id in basins_in_cluster(cl):
            info = GAUGE_INFO_FRAC[gauge_id]
            rgi_ids = info.get('rgi_ids', [])
            mask_ids = [r for r in rgi_ids if r in rgi_all]
            if not mask_ids:
                continue
            mask = np.isin(rgi_all, mask_ids)
            basin_snow = snow_annual[:, mask].sum(axis=1)
            basin_melt = melt_annual[:, mask].sum(axis=1)

            ann = pd.DataFrame({'year': actual_years, 'snow_m3': basin_snow, 'melt_m3': basin_melt})
            base = ann[ann['year'].isin(BASELINE)]
            drgt = ann[ann['year'].isin(range(2010, 2020))]
            if len(base) == 0 or len(drgt) == 0:
                continue

            snow_base, melt_base = base['snow_m3'].mean(), base['melt_m3'].mean()
            snow_dr, melt_dr     = drgt['snow_m3'].mean(), drgt['melt_m3'].mean()
            mb_base, mb_dr = snow_base - melt_base, snow_dr - melt_dr
            delta = mb_dr - mb_base
            if abs(delta) < 0.1 * abs(mb_base):
                continue

            mb_cf_snow = snow_dr   - melt_base
            mb_cf_melt = snow_base - melt_dr
            pct_accum = (mb_cf_snow - mb_base) / delta * 100
            pct_ablat = (mb_cf_melt - mb_base) / delta * 100

            accum_ablat_records.append({
                'gauge_id': gauge_id, 'cluster': cl, 'dataset': ds_name,
                'snow_chg_pct': (snow_dr-snow_base)/snow_base*100 if snow_base>0 else np.nan,
                'melt_chg_pct': (melt_dr-melt_base)/melt_base*100 if melt_base>0 else np.nan,
                'delta_mb_m3': delta,
                'pct_accum_driven': np.clip(pct_accum, -200, 200),
                'pct_ablat_driven': np.clip(pct_ablat, -200, 200),
            })

df_mb_decomp = pd.DataFrame(accum_ablat_records)
df_mb_decomp.to_csv(f'{OUT_DIR}/section1_5_mb_decomposition.csv', index=False)

print('Section 1.4: MB change decomposition -- accumulation vs. ablation driven')
print('(worsening-MB basins only, i.e. delta_mb < 0; median across basins)')
print(f'{"Cluster":6s} {"Dataset":8s} {"n":4s} {"Snow d%":9s} {"Melt d%":9s}  {"% accum-driven":15s} {"% ablat-driven"}')
print('-'*90)
for cl in CLUSTERS:
    for ds_name in datasets:
        sub = df_mb_decomp[(df_mb_decomp['cluster']==cl) & (df_mb_decomp['dataset']==ds_name) & (df_mb_decomp['delta_mb_m3']<0)]
        if len(sub) == 0:
            continue
        print(f'{cl:6s} {ds_name:8s} {len(sub):4d} {sub["snow_chg_pct"].median():+8.1f}% '
              f'{sub["melt_chg_pct"].median():+8.1f}%  {sub["pct_accum_driven"].median():14.1f}%  '
              f'{sub["pct_ablat_driven"].median():14.1f}%')

print('\nSection 1 complete -- proceed to Section 1.5 (lag detection)')

### Interpreting the accumulation/ablation decomposition -- what "less snow" actually means physically

Across most of Chile's glacierized regions (DA2, DA3, WA1, WA2, and
parts of WA3/WA4/WA6), the median decomposition above shows mass-balance
decline is primarily **accumulation-driven** (falling snowfall), with
melt change often small. This should NOT be read as "melt doesn't
matter" -- the correct physical framing is:

**A glacier's ice only shrinks when melt exceeds new snow input in a
given year.** If melt stays roughly the same from year to year but
snowfall declines, the *same amount of melt* now has less fresh snow to
consume before it starts eating into the glacier's underlying ice
reserve. In other words: declining accumulation doesn't mean the
glacier is "safe" from melt -- it means the snow buffer that used to
absorb that melt is thinner, so the same melt energy converts directly
to ice loss instead. The glacier shrinks either way; a declining-snow
regime and an increasing-melt regime just get there through different
mechanisms.

**DA1 is the clear exception and the more acute case**: there, MB
decline is 84-99% *ablation*-driven -- melt itself is increasing, not
just snow declining. That's a genuine warming/melt-rate signal on top of
the ice-reserve mechanism described above, and it's consistent with
DA1 also showing the most dramatic streamflow-contribution changes in
Section 2 below (pending the >100% peak-month fix -- see Cell 0b).

In [ ]:
# Section 1.5: Deseasonalized lag detection, ALL THREE datasets. Feeds Section 2. 
def deseasonalize(series, months):
    anom = series.copy().astype(float)
    for mo in range(1, 13):
        mask = months == mo
        if mask.sum() > 0:
            anom[mask] -= anom[mask].mean()
    return anom

def best_lag_pearson(x, y, max_lag=6):
    best_r, best_lag = 0, 0
    for lag in range(0, max_lag + 1):
        if lag == 0:
            valid = ~(np.isnan(x) | np.isnan(y))
            if valid.sum() < 10:
                continue
            r, _ = stats.pearsonr(x[valid], y[valid])
        else:
            xlag, ylag = x[:-lag], y[lag:]
            valid = ~(np.isnan(xlag) | np.isnan(ylag))
            if valid.sum() < 10:
                continue
            r, _ = stats.pearsonr(xlag[valid], ylag[valid])
        if abs(r) > abs(best_r):
            best_r, best_lag = r, lag
    return best_lag, best_r

print('Section 1.5: Deseasonalized lag detection by cluster and dataset')
print('='*80)

DESEAS_LAG_BY_DS = {}
for ds_name in datasets:
    print(f'\n--- {ds_name} ---')
    print(f'{"Cluster":6s} {"n":4s} {"Raw median":11s} {"Deseas median":13s} {"Interpretation"}')
    print('-'*70)
    DESEAS_LAG_BY_DS[ds_name] = {}
    for cl in CLUSTERS:
        cl_gauges = basins_in_cluster(cl)
        cl_raw, cl_des = [], []
        for gauge_id in cl_gauges:
            res = RESULTS[ds_name].get(gauge_id)
            if res is None:
                continue
            m  = res['monthly']
            cy = GAUGE_INFO_FRAC[gauge_id].get('complete_years', YEARS)
            mc = (m[m['year'].isin(cy)].sort_values(['year','month']).dropna(subset=['q_m3_mon']).reset_index(drop=True))
            if len(mc) < 36:
                continue
            glac   = mc['glacier_runoff_m3'].values.astype(float)
            q      = mc['q_m3_mon'].values.astype(float)
            months = mc['month'].values
            lag_r, _ = best_lag_pearson(glac, q)
            lag_d, _ = best_lag_pearson(deseasonalize(glac, months), deseasonalize(q, months))
            cl_raw.append(lag_r)
            cl_des.append(lag_d)

        if not cl_raw:
            DESEAS_LAG_BY_DS[ds_name][cl] = 0
            continue
        med_r, med_d = np.median(cl_raw), np.median(cl_des)
        lag_final = int(round(med_d))
        DESEAS_LAG_BY_DS[ds_name][cl] = lag_final
        interp = 'ALIASING -- raw lag not used' if med_d < med_r - 1 else f'Genuine {lag_final}-month delay applied'
        print(f'{cl:6s} {len(cl_raw):4d} {med_r:11.1f} {med_d:13.1f} {interp}')

print()
print('DESEAS_LAG_BY_DS:')
for ds_name in datasets:
    print(f'  {ds_name}: {DESEAS_LAG_BY_DS[ds_name]}')

print()
print('Cross-dataset agreement check -- does the lag conclusion (aliasing vs. genuine)')
print('depend on which dataset detected it?')
for cl in CLUSTERS:
    vals = [DESEAS_LAG_BY_DS[ds][cl] for ds in datasets]
    agree = len(set(vals)) == 1
    print(f'  {cl:6s}: {dict(zip(datasets, vals))}  {"AGREE" if agree else "DISAGREE"}')

"Aliasing" here means the raw cross-correlation was picking up a fake
lag caused by the fact that both glacier melt and streamflow follow the
same annual seasonal cycle -- melt peaks in summer, streamflow peaks in
summer -- so a naive correlation between them can show a spurious
multi-month "lag" just because shifting one series by a few months still
lines its seasonal bump up reasonably well with the other series'
bump from a different year, not because there's a real physical delay.
The deseasonalizing step subtracts each month's own long-term average
before correlating, removing that shared seasonal shape and leaving
only genuine month-to-month covariation. Whenever the deseasonalized
lag collapses to 0 while the raw lag was large (WA1: 5.5 to 0.0, DA1:
2.0 to 0.0, DA3: 2.0 to 0.0), that's the signature of aliasing.
Clusters where the deseasonalized lag stays close to the raw lag (DA2:
1.5 to 1.0, WA2 through WA6: values staying around 1.0) show a genuine,
physically real routing delay that survives deseasonalizing.

In [ ]:
# Section 1.5b (build): Climate driver anomalies by cluster
def load_camels_mon_full(fname):
    df = pd.read_csv(f'{CAMELS_FULL}/{fname}')
    df = df[(df['year']>=2000)&(df['year']<=2019)].copy()
    gauge_cols = [c for c in df.columns if c not in ['date','year','month','day']]
    return df[['year','month']+gauge_cols]

precip_clim = load_camels_mon_full('precip_cr2met_mm_mon.csv')
tmean_clim  = load_camels_mon_full('tmean_cr2met_C_mon.csv')
pet_clim    = load_camels_mon_full('pet_hargreaves_mm_mon.csv')

clim_records = []
for cl in CLUSTERS:
    gauges = [g for g in basins_in_cluster(cl) if g in precip_clim.columns]
    if not gauges:
        continue
    for year in YEARS:
        p_ann = precip_clim[precip_clim['year']==year][gauges].sum().mean()
        t_ann = tmean_clim[tmean_clim['year']==year][gauges].mean().mean()
        e_ann = pet_clim[pet_clim['year']==year][gauges].sum().mean()
        clim_records.append({'cluster': cl, 'year': year, 'precip_ann': p_ann, 'tmean_ann': t_ann, 'pet_ann': e_ann})

df_clim = pd.DataFrame(clim_records)
baseline_means = (df_clim[df_clim['year'].isin(BASELINE)].groupby('cluster')[['precip_ann','tmean_ann','pet_ann']]
                  .mean().rename(columns=lambda c: c+'_base'))
df_clim = df_clim.merge(baseline_means, on='cluster', how='left')
df_clim['precip_anom_pct'] = (df_clim['precip_ann']-df_clim['precip_ann_base'])/df_clim['precip_ann_base']*100
df_clim['tmean_anom']      = df_clim['tmean_ann']-df_clim['tmean_ann_base']
df_clim['pet_anom_pct']    = (df_clim['pet_ann']-df_clim['pet_ann_base'])/df_clim['pet_ann_base']*100

print(f'df_clim built: {df_clim.shape}')

In [ ]:
# Section 1.5b (plot): Climate driver anomalies figure
from matplotlib.colors import TwoSlopeNorm
from matplotlib.cm import ScalarMappable

drought_yrs = range(2010, 2020)

var_config = {
    'precip_anom_pct': {'label': 'Precip anomaly (%)', 'cmap': 'RdBu', 'title': 'Precipitation (red = drier, blue = wetter)'},
    'tmean_anom': {'label': 'Temp anomaly (C)', 'cmap': 'coolwarm', 'title': 'Temperature (blue = cooler, red = warmer)'},
    'pet_anom_pct': {'label': 'PET anomaly (%)', 'cmap': 'BrBG_r', 'title': 'PET (green = less evaporative demand, brown = more)'},
}

fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
for ax, (var, cfg) in zip(axes, var_config.items()):
    vals = [df_clim[(df_clim['cluster']==cl)&(df_clim['year'].isin(drought_yrs))][var].mean() for cl in CLUSTERS]
    vmax_abs = max(abs(min(vals)), abs(max(vals)))
    norm = TwoSlopeNorm(vmin=-vmax_abs, vcenter=0, vmax=vmax_abs)
    cmap = plt.get_cmap(cfg['cmap'])
    bar_colors = [cmap(norm(v)) for v in vals]

    bars = ax.bar(CLUSTERS, vals, color=bar_colors, edgecolor='k', lw=0.6)
    ax.axhline(0, color='k', lw=1)
    for bar, v in zip(bars, vals):
        va = 'bottom' if v >= 0 else 'top'
        ax.text(bar.get_x()+bar.get_width()/2, v, f'{v:+.1f}', ha='center', va=va, fontsize=7)

    ax.set_ylabel(cfg['label'], fontsize=10)
    ax.set_title(cfg['title'], fontsize=10, fontweight='bold')
    ax.tick_params(axis='x', labelsize=8)
    ax.grid(True, alpha=0.3, axis='y')
    sm = ScalarMappable(cmap=cmap, norm=norm)
    cbar = fig.colorbar(sm, ax=ax, orientation='horizontal', pad=0.15, shrink=0.85)
    cbar.ax.tick_params(labelsize=7)

fig.suptitle('Section 1.5b: Climate Driver Anomalies, 2010-2019 vs 2000-2009 Baseline', fontsize=13, y=1.05)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/section1_5b_climate_drivers.png', dpi=150, bbox_inches='tight')
plt.show()

# SECTION 2 -- Q2: Glacier contribution to streamflow (lag-corrected)

In [ ]:
# Build lag-corrected monthly dataset (used by all of Section 2)
def get_lag_corrected_monthly(gauge_id, ds_name, cluster):
    """Returns monthly df with glac_shifted = glacier_runoff_m3 shifted by
    DESEAS_LAG[cluster] months. lag=0 clusters are returned unshifted."""
    res = RESULTS[ds_name].get(gauge_id)
    if res is None:
        return None
    m = res['monthly'].copy()
    cy = GAUGE_INFO_FRAC[gauge_id].get('complete_years', YEARS)
    m = m[m['year'].isin(cy)].sort_values(['year','month']).reset_index(drop=True)
    lag = DESEAS_LAG_BY_DS[ds_name].get(cluster, 0)
    m['glac_shifted'] = (m['glacier_runoff_m3'].shift(lag) if lag > 0 else m['glacier_runoff_m3'])
    return m

In [ ]:
# Compute actual peak glacier-fraction month per cluster, don't assume March
print('Peak glacier contribution month by cluster (CR2MET, mean fraction by month)')
print('='*70)

month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
PEAK_MONTH = {}

for cl in CLUSTERS:
    gauges_in_cl = basins_in_cluster(cl)
    mon_fracs = []
    for gid in gauges_in_cl:
        m = get_lag_corrected_monthly(gid, 'CR2MET', cl)
        if m is None:
            continue
        mf = m.groupby('month').apply(
            lambda g: g['glac_shifted'].sum() / g['q_m3_mon'].sum() if g['q_m3_mon'].sum() > 0 else np.nan)
        mon_fracs.append(mf)
    if not mon_fracs:
        continue
    mean_mf = pd.concat(mon_fracs, axis=1).mean(axis=1)
    peak_month = mean_mf.idxmax()
    PEAK_MONTH[cl] = peak_month
    print(f'{cl:6s} peak month = {month_labels[peak_month-1]:4s} ({mean_mf[peak_month]*100:.1f}%)')

print(f'\nPEAK_MONTH dict: {PEAK_MONTH}')
print()
print('*** SEE CELL 0b (duplicate-basin diagnostic) before trusting any peak-')
print('*** month percentage below -- DA1 and DA3 previously showed >100%')
print('*** peak-month fractions, a physically impossible value most likely')
print('*** caused by duplicated/nested CAMELS basins along the same river')
print('*** (e.g. Rio Copiapo gauged at 3 points, each inheriting the SAME')
print('*** glacier melt total but different, sometimes tiny, streamflow).')

In [ ]:
# Peak ABSOLUTE glacier melt month per cluster (not fraction of streamflow)

print('Peak absolute glacier melt month by cluster (CR2MET)')
print('='*70)

PEAK_MELT_MONTH = {}
for cl in CLUSTERS:
    gauges_in_cl = basins_in_cluster(cl)
    mon_melt = []
    for gid in gauges_in_cl:
        res = RESULTS['CR2MET'].get(gid)
        if res is None:
            continue
        m = res['monthly']
        cy = GAUGE_INFO_FRAC[gid].get('complete_years', YEARS)
        m = m[m['year'].isin(cy)]
        mm = m.groupby('month')['melt_on_glacier_m3'].sum()
        mon_melt.append(mm)
    if not mon_melt:
        continue
    total_melt_by_month = pd.concat(mon_melt, axis=1).sum(axis=1)
    peak_month = total_melt_by_month.idxmax()
    PEAK_MELT_MONTH[cl] = peak_month
    print(f'{cl:6s} peak melt month = {month_labels[peak_month-1]:4s}')

print(f'\nPEAK_MELT_MONTH dict: {PEAK_MELT_MONTH}')

In [ ]:
# National + cluster summary table: annual, FMA, peak month (per-cluster)
def basin_mean_frac(gauge_id, ds_name, cluster, months=None):
    m = get_lag_corrected_monthly(gauge_id, ds_name, cluster)
    if m is None:
        return np.nan
    if months is not None:
        m = m[m['month'].isin(months)]
    m = m.dropna(subset=['glac_shifted', 'q_m3_mon'])
    m = m[m['q_m3_mon'] > 0]
    if len(m) == 0:
        return np.nan
    ann = m.groupby('year').agg(glac=('glac_shifted','sum'), q=('q_m3_mon','sum'))
    ann = ann[ann['q'] > 0]
    if len(ann) == 0:
        return np.nan
    return (ann['glac'] / ann['q']).mean() * 100   # mean of per-year ratios (correct convention)

month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

print('Section 2a: Glacier fraction of streamflow -- lag-corrected')
print('Peak month is CLUSTER-SPECIFIC (from PEAK_MONTH dict), not fixed to March')
print('='*95)
print(f'{"Scope":20s} {"Dataset":8s} {"n_basins":9s} {"Annual %":9s} {"FMA %":8s} {"Peak mo %":9s} {"(month)"}')
print('-'*95)

section2a_records = []
for cl in ['NATIONAL'] + CLUSTERS:
    gauges = (list(GAUGE_INFO_FRAC.keys()) if cl == 'NATIONAL' else basins_in_cluster(cl))
    peak_mo = PEAK_MONTH.get(cl, 2)
    peak_mo_label = month_labels[peak_mo - 1]

    for ds_name in datasets:
        ann_vals  = [basin_mean_frac(g, ds_name, cl, None) for g in gauges]
        fma_vals  = [basin_mean_frac(g, ds_name, cl, FMA_MONTHS) for g in gauges]
        peak_vals = [basin_mean_frac(g, ds_name, cl, [peak_mo]) for g in gauges]
        ann_vals  = [v for v in ann_vals if not np.isnan(v)]
        fma_vals  = [v for v in fma_vals if not np.isnan(v)]
        peak_vals = [v for v in peak_vals if not np.isnan(v)]
        if not ann_vals:
            continue
        section2a_records.append({
            'scope': cl, 'dataset': ds_name, 'n_basins': len(ann_vals),
            'annual_pct': np.mean(ann_vals), 'annual_pct_sd': np.std(ann_vals),
            'fma_pct': np.mean(fma_vals) if fma_vals else np.nan,
            'peak_month': peak_mo, 'peak_month_label': peak_mo_label,
            'peak_month_pct': np.mean(peak_vals) if peak_vals else np.nan,
        })
        print(f'{cl:20s} {ds_name:8s} {len(ann_vals):9d} {np.mean(ann_vals):8.1f}% '
              f'{np.mean(fma_vals) if fma_vals else np.nan:7.1f}% '
              f'{np.mean(peak_vals) if peak_vals else np.nan:8.1f}%  ({peak_mo_label})')

df_section2a = pd.DataFrame(section2a_records)
df_section2a.to_csv(f'{OUT_DIR}/section2a_contribution_summary.csv', index=False)

n_over_100 = df_section2a[df_section2a['peak_month_pct'] > 100]
if len(n_over_100):
    print()
    print(f'*** WARNING: {len(n_over_100)} scope/dataset rows show peak-month % > 100 ***')
    print(n_over_100[['scope','dataset','peak_month_pct']].to_string(index=False))
    print('Run Cell 0b before citing any of these numbers.')

Nationally, glacier melt contributes ~7-12% of annual streamflow
depending on climate dataset, rising to ~15-25% in peak ablation season
(FMA), and higher still in each region's actual peak month. **DA1 and
DA2's peak-month numbers should be treated as unverified pending Cell
0b's duplicate-basin check** -- some rows exceed the physically valid
100% ceiling, the same issue that also appeared in DA3 in an earlier
pass. This is not just a DA1/DA3 quirk; it needs to be traced (nested
basin duplication is the leading hypothesis) and resolved before any
peak-month number is cited in the paper.

In [ ]:
# Monthly seasonality by region (lag-corrected)
fig, axes = plt.subplots(2, 5, figsize=(18, 8), sharey=False)
axes_flat = axes.flatten()
month_labels_short = ['J','F','M','A','M','J','J','A','S','O','N','D']

for ax, cl in zip(axes_flat, CLUSTERS):
    gauges_in_cl = basins_in_cluster(cl)
    for ds_name in datasets:
        mon_fracs = []
        for gid in gauges_in_cl:
            m = get_lag_corrected_monthly(gid, ds_name, cl)
            if m is None:
                continue
            mf = m.groupby('month').apply(
                lambda g: g['glac_shifted'].sum() / g['q_m3_mon'].sum() if g['q_m3_mon'].sum() > 0 else np.nan)
            mon_fracs.append(mf)
        if mon_fracs:
            mean_mf = pd.concat(mon_fracs, axis=1).mean(axis=1)
            ax.plot(mean_mf.index, mean_mf.values*100, color=DS_COLORS[ds_name], lw=1.8, label=ds_name)

    ax.set_title(f'{cl} -- {REGION_LABELS[cl]}', fontsize=9)
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(month_labels_short, fontsize=7)
    ax.grid(True, alpha=0.3)

axes_flat[0].set_ylabel('Glacier runoff fraction (%)')
axes_flat[5].set_ylabel('Glacier runoff fraction (%)')
handles = [plt.Line2D([0],[0], color=DS_COLORS[d], lw=2, label=d) for d in datasets]
fig.legend(handles=handles, loc='lower center', ncol=3, fontsize=10)
fig.suptitle('Section 2b: Mean Monthly Glacier Runoff Fraction by Region (Lag-Corrected)\n'
             '2000-2019 average, per cluster', fontsize=12)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig(f'{OUT_DIR}/section2b_monthly_seasonality_by_region.png', dpi=150, bbox_inches='tight')
plt.show()

WA5 has no monitored streamflow at all in the CAMELS dataset -- hence the blank panel.

## Section 2c: Pre- vs. post-2010 comparison (calendar), annual + JFM

Fractions are the mean of each year's own ratio (glacier runoff / streamflow), lag-corrected, matching Sections 2a and 3a. `period_frac` is also used by the maps below.

In [ ]:
JFM_MONTHS = [1, 2, 3]   # captures every cluster's peak month: DA1=Jan, 7 clusters=Feb, DA2=Mar

def period_frac(gauge_id, ds_name, cluster, years, months=None):
    """Mean over years of (glacier runoff / streamflow) in %, lag-corrected.
    Mean of per-year ratios, NOT sum(glac)/sum(q)."""
    m = get_lag_corrected_monthly(gauge_id, ds_name, cluster)
    if m is None:
        return np.nan
    m = m[m['year'].isin(years)]
    if months is not None:
        m = m[m['month'].isin(months)]
    m = m.dropna(subset=['glac_shifted', 'q_m3_mon'])
    m = m[m['q_m3_mon'] > 0]
    if len(m) == 0:
        return np.nan
    ann = m.groupby('year').agg(glac=('glac_shifted', 'sum'), q=('q_m3_mon', 'sum'))
    ann = ann[ann['q'] > 0]
    if len(ann) == 0:
        return np.nan
    return (ann['glac'] / ann['q']).mean() * 100

print('Section 2c: 2000-2009 vs 2010-2019, annual + JFM, lag-corrected, mean of yearly ratios')
print('Paired Wilcoxon signed-rank test per cluster')
print('='*100)

section2c_all = []
for cl in CLUSTERS:
    gauges = basins_in_cluster(cl)
    for scope, months in [('Annual', None), ('JFM', JFM_MONTHS)]:
        for ds_name in datasets:
            base_vals, dr_vals = [], []
            for gid in gauges:
                b = period_frac(gid, ds_name, cl, BASELINE, months)
                d = period_frac(gid, ds_name, cl, range(2010, 2020), months)
                if not np.isnan(b) and not np.isnan(d):
                    base_vals.append(b); dr_vals.append(d)
            if len(base_vals) < 3:
                continue
            try:
                stat, p = stats.wilcoxon(dr_vals, base_vals)
            except ValueError:
                p = np.nan
            section2c_all.append({
                'cluster': cl, 'scope': scope, 'dataset': ds_name, 'n': len(base_vals),
                'base_pct': np.mean(base_vals), 'drought_pct': np.mean(dr_vals),
                'delta': np.mean(dr_vals) - np.mean(base_vals), 'p': p, 'sig': sig_stars(p),
            })

df_section2c = pd.DataFrame(section2c_all)
df_section2c.to_csv(f'{DROUGHT_DIR}/section2c_calendar_comparison.csv', index=False)

for ds_name in datasets:
    print(f'\n{ds_name}, JFM scope:')
    print(df_section2c[(df_section2c['dataset'] == ds_name) & (df_section2c['scope'] == 'JFM')]
          [['cluster', 'n', 'base_pct', 'drought_pct', 'delta', 'p', 'sig']].round(3).to_string(index=False))

print('\nAny base_pct or drought_pct above 100 means modeled glacier runoff is greater than streamflow')
print('in that cluster; check notebook 03 (cell 17b, runoff > streamflow) before citing it.')

**Maps.** The next three cells draw the Chile maps (Section 2a with pie pullouts, the 3-panel version, and the megadrought-only map). They depend on each other and on the cells above, so run them in order.

In [ ]:
# Section 2a figure: Chile map, respaced pullouts to avoid label overlap
import geopandas as gpd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.patches import Wedge, Patch
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, to_hex

bounds = gpd.read_file(BOUNDS_PATH).to_crs(epsg=4326)
bounds['gauge_id'] = bounds['gauge_id'].astype(str)

REGION_ANCHORS = {
    'OT3': (-68.8, -19.5), 'DA1': (-69.5, -27.5), 'DA2': (-70.2, -32.5), 'DA3': (-70.8, -35.2),
    'WA1': (-71.5, -37.5), 'WA2': (-72.5, -42.0), 'WA3': (-73.0, -46.5), 'WA4': (-73.5, -50.0),
    'WA6': (-68.5, -54.5),
}
PULLOUT_POS = {
    'OT3': (-62.5, -18.5), 'DA1': (-62.5, -24.5), 'DA2': (-62.5, -30.5), 'DA3': (-62.5, -34.5),
    'WA1': (-62.5, -38.5), 'WA2': (-62.5, -42.5), 'WA3': (-62.5, -46.5), 'WA4': (-62.5, -50.5),
    'WA6': (-62.5, -55.0),
}

cluster_basin_areas = {cl: sum(GAUGE_INFO_FRAC[g]['basin_area_km2'] for g in basins_in_cluster(cl))
                       for cl in CLUSTERS if cl != 'WA5'}
area_max = max(cluster_basin_areas.values())
CIRC_MAX, CIRC_MIN = 1.3, 0.45

def area_to_radius(area):
    if area <= 0:
        return CIRC_MIN
    return CIRC_MIN + (area / area_max) ** 0.5 * (CIRC_MAX - CIRC_MIN)

def regional_frac(cluster, months=None):
    fracs = []
    for gid in basins_in_cluster(cluster):
        f = basin_mean_frac(gid, 'CR2MET', cluster, months)
        if not np.isnan(f):
            fracs.append(f)
    return np.nanmean(fracs) if fracs else np.nan

for scope, months in [('Annual', None), ('JFM', JFM_MONTHS)]:
    col = f'frac_{scope}'
    bounds[col] = bounds['gauge_id'].apply(
        lambda g: basin_mean_frac(g, 'CR2MET', GAUGE_INFO_FRAC[g]['clusters'][0], months)
        if g in GAUGE_INFO_FRAC else np.nan)

bounds['_area_deg2'] = bounds.geometry.area
bounds = bounds.sort_values('_area_deg2', ascending=False).reset_index(drop=True)

map_vmax = np.nanpercentile(bounds['frac_JFM'].dropna(), 95)
map_norm = Normalize(vmin=0, vmax=map_vmax)
cmap_map = plt.cm.YlOrRd
GREY, GREY_EDGE, BLUE_EDGE = '#e0e0e0', '#aaaaaa', '#555555'

try:
    world = gpd.read_file('https://naturalearth.s3.amazonaws.com/50m_cultural/ne_50m_admin_0_countries.zip')
    chile, arg = world[world['NAME']=='Chile'], world[world['NAME']=='Argentina']
    has_borders = True
except Exception:
    has_borders = False

fig = plt.figure(figsize=(22, 11))
gs = fig.add_gridspec(1, 4, width_ratios=[1,1,1,0.05], wspace=0.05)

for c_idx, (scope, months) in enumerate([('Annual', None), ('JFM', JFM_MONTHS)]):
    ax = fig.add_subplot(gs[0, c_idx])
    if has_borders:
        chile.plot(ax=ax, facecolor='#f2f2f2', edgecolor='#999999', lw=0.5, zorder=0)
        arg.plot(ax=ax, facecolor='#fafafa', edgecolor='#999999', lw=0.5, zorder=0)

    col = f'frac_{scope}'
    vals = bounds[col].values
    face = [to_hex(cmap_map(map_norm(v))) if not np.isnan(v) else GREY for v in vals]
    edge = [BLUE_EDGE if not np.isnan(v) else GREY_EDGE for v in vals]
    bounds.plot(ax=ax, color=face, edgecolor=edge, linewidth=0.25, zorder=2)

    for cl in CLUSTERS:
        if cl == 'WA5':
            continue
        mean_frac = regional_frac(cl, months)
        anchor, pullout = REGION_ANCHORS[cl], PULLOUT_POS[cl]
        radius = area_to_radius(cluster_basin_areas[cl])

        if np.isnan(mean_frac) or mean_frac == 0:
            ax.add_patch(plt.Circle(pullout, radius, facecolor=GREY, edgecolor='#888888', lw=0.8, zorder=6))
        else:
            frac_clip = min(mean_frac/100, 1.0)
            theta1 = 90
            for size, fc in [(frac_clip, '#e74c3c'), (1-frac_clip, '#ecf0f1')]:
                theta2 = theta1 - size*360
                ax.add_patch(Wedge(pullout, radius, min(theta1,theta2), max(theta1,theta2),
                            facecolor=fc, edgecolor='#555555', lw=0.6, zorder=6))
                theta1 = theta2
            ax.add_patch(plt.Circle(pullout, radius, facecolor='none', edgecolor='#333333', lw=0.8, zorder=7))
            ax.text(pullout[0], pullout[1]-radius-0.3, f'{mean_frac:.1f}%', ha='center', va='top',
                   fontsize=7.5, fontweight='bold', color='#1a1a1a', zorder=8,
                   bbox=dict(boxstyle='round,pad=0.15', facecolor='white', edgecolor='none', alpha=0.75))

        ax.text(pullout[0], pullout[1]+radius+0.2, cl, ha='center', va='bottom', fontsize=7, fontweight='bold', zorder=8)
        ax.annotate('', xy=anchor, xytext=(pullout[0]-radius*0.1, pullout[1]),
                   arrowprops=dict(arrowstyle='-', color='#666666', lw=0.7), zorder=5)

    ax.set_xlim(-76, -60); ax.set_ylim(-56, -17); ax.set_aspect('equal')
    ax.set_title(f'{scope}', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.15, lw=0.3)

cbar_ax = fig.add_subplot(gs[0, 3])
sm = ScalarMappable(cmap=cmap_map, norm=map_norm)
fig.colorbar(sm, cax=cbar_ax).set_label('Glacier fraction of streamflow (%)', fontsize=9)

fig.legend(handles=[
    Patch(facecolor='#e74c3c', edgecolor='#555555', label='Glacier melt fraction'),
    Patch(facecolor='#ecf0f1', edgecolor='#555555', label='Remaining streamflow'),
    Patch(facecolor=GREY, edgecolor=GREY_EDGE, label='No data'),
], loc='lower center', ncol=3, fontsize=9, bbox_to_anchor=(0.45, -0.02), framealpha=0.9)

fig.suptitle('Section 2a: Glacier Contribution to Streamflow -- Chile (CR2MET, 2000-2019)', fontsize=13, y=1.05)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/section2a_map_piecharts.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Section 2a figure: Chile map, 3-panel (Annual full-period, JFM Megadrought
# 2010-2019, JFM full-period 2000-2019), respaced pullouts to avoid overlap
import geopandas as gpd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.patches import Wedge, Patch
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, to_hex

bounds = gpd.read_file(BOUNDS_PATH).to_crs(epsg=4326)
bounds['gauge_id'] = bounds['gauge_id'].astype(str)

REGION_ANCHORS = {
    'OT3': (-68.8, -19.5), 'DA1': (-69.5, -27.5),
    'DA2': (-70.2, -32.5), 'DA3': (-70.8, -35.2),
    'WA1': (-71.5, -37.5), 'WA2': (-72.5, -42.0),
    'WA3': (-73.0, -46.5), 'WA4': (-73.5, -50.0),
    'WA6': (-68.5, -54.5),
}
PULLOUT_POS = {
    'OT3': (-62.5, -18.5), 'DA1': (-62.5, -24.5),
    'DA2': (-62.5, -30.5), 'DA3': (-62.5, -34.5),
    'WA1': (-62.5, -38.5), 'WA2': (-62.5, -42.5),
    'WA3': (-62.5, -46.5), 'WA4': (-62.5, -50.5),
    'WA6': (-62.5, -55.0),
}

cluster_basin_areas = {
    cl: sum(GAUGE_INFO_FRAC[g]['basin_area_km2'] for g in basins_in_cluster(cl))
    for cl in CLUSTERS if cl != 'WA5'
}
area_max = max(cluster_basin_areas.values())
CIRC_MAX, CIRC_MIN = 1.3, 0.45

def area_to_radius(area):
    if area <= 0:
        return CIRC_MIN
    return CIRC_MIN + (area / area_max) ** 0.5 * (CIRC_MAX - CIRC_MIN)

def regional_frac(cluster, months=None, years=None):
    fracs = []
    for gid in basins_in_cluster(cluster):
        if years is not None:
            f = period_frac(gid, 'CR2MET', cluster, years, months)
        else:
            f = basin_mean_frac(gid, 'CR2MET', cluster, months)
        if not np.isnan(f):
            fracs.append(f)
    return np.nanmean(fracs) if fracs else np.nan

# Build per-basin fraction columns: Annual (full), JFM Megadrought (2010-19),
# JFM full period (2000-19)
bounds['frac_Annual'] = bounds['gauge_id'].apply(
    lambda g: basin_mean_frac(g, 'CR2MET', GAUGE_INFO_FRAC[g]['clusters'][0], None)
    if g in GAUGE_INFO_FRAC else np.nan)
bounds['frac_JFM_drought'] = bounds['gauge_id'].apply(
    lambda g: period_frac(g, 'CR2MET', GAUGE_INFO_FRAC[g]['clusters'][0],
                          range(2010,2020), JFM_MONTHS)
    if g in GAUGE_INFO_FRAC else np.nan)
bounds['frac_JFM_full'] = bounds['gauge_id'].apply(
    lambda g: basin_mean_frac(g, 'CR2MET', GAUGE_INFO_FRAC[g]['clusters'][0], JFM_MONTHS)
    if g in GAUGE_INFO_FRAC else np.nan)

bounds['_area_deg2'] = bounds.geometry.area
bounds = bounds.sort_values('_area_deg2', ascending=False).reset_index(drop=True)

map_vmax = np.nanpercentile(bounds['frac_JFM_full'].dropna(), 95)
map_norm = Normalize(vmin=0, vmax=map_vmax)
cmap_map = plt.cm.YlOrRd
GREY, GREY_EDGE, BLUE_EDGE = '#e0e0e0', '#aaaaaa', '#555555'

try:
    world = gpd.read_file(
        'https://naturalearth.s3.amazonaws.com/50m_cultural/ne_50m_admin_0_countries.zip')
    chile, arg = world[world['NAME']=='Chile'], world[world['NAME']=='Argentina']
    has_borders = True
except Exception:
    has_borders = False

fig = plt.figure(figsize=(28, 11))
gs = fig.add_gridspec(1, 5, width_ratios=[1,1,1,1,0.05], wspace=0.05)

panels = [
    ('Annual\n(full period)', 'frac_Annual', None, None),
    ('JFM, Megadrought\n(2010-2019)', 'frac_JFM_drought', JFM_MONTHS, range(2010,2020)),
    ('JFM\n(full period, 2000-2019)', 'frac_JFM_full', JFM_MONTHS, None),
]

for c_idx, (title, col, months, years) in enumerate(panels):
    ax = fig.add_subplot(gs[0, c_idx])
    if has_borders:
        chile.plot(ax=ax, facecolor='#f2f2f2', edgecolor='#999999', lw=0.5, zorder=0)
        arg.plot(ax=ax, facecolor='#fafafa', edgecolor='#999999', lw=0.5, zorder=0)

    vals = bounds[col].values
    face = [to_hex(cmap_map(map_norm(v))) if not np.isnan(v) else GREY for v in vals]
    edge = [BLUE_EDGE if not np.isnan(v) else GREY_EDGE for v in vals]
    bounds.plot(ax=ax, color=face, edgecolor=edge, linewidth=0.25, zorder=2)

    for cl in CLUSTERS:
        if cl == 'WA5':
            continue
        mean_frac = regional_frac(cl, months, years)
        anchor, pullout = REGION_ANCHORS[cl], PULLOUT_POS[cl]
        radius = area_to_radius(cluster_basin_areas[cl])

        if np.isnan(mean_frac) or mean_frac == 0:
            ax.add_patch(plt.Circle(pullout, radius, facecolor=GREY,
                        edgecolor='#888888', lw=0.8, zorder=6))
        else:
            frac_clip = min(mean_frac/100, 1.0)
            theta1 = 90
            for size, fc in [(frac_clip, '#e74c3c'), (1-frac_clip, '#ecf0f1')]:
                theta2 = theta1 - size*360
                ax.add_patch(Wedge(pullout, radius, min(theta1,theta2), max(theta1,theta2),
                            facecolor=fc, edgecolor='#555555', lw=0.6, zorder=6))
                theta1 = theta2
            ax.add_patch(plt.Circle(pullout, radius, facecolor='none',
                        edgecolor='#333333', lw=0.8, zorder=7))
            ax.text(pullout[0], pullout[1]-radius-0.3, f'{mean_frac:.1f}%',
                   ha='center', va='top', fontsize=7.5, fontweight='bold',
                   color='#1a1a1a', zorder=8,
                   bbox=dict(boxstyle='round,pad=0.15', facecolor='white',
                            edgecolor='none', alpha=0.75))

        ax.text(pullout[0], pullout[1]+radius+0.2, cl, ha='center', va='bottom',
               fontsize=7, fontweight='bold', zorder=8)
        ax.annotate('', xy=anchor, xytext=(pullout[0]-radius*0.1, pullout[1]),
                   arrowprops=dict(arrowstyle='-', color='#666666', lw=0.7), zorder=5)

    ax.set_xlim(-76, -60); ax.set_ylim(-56, -17); ax.set_aspect('equal')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.15, lw=0.3)

cbar_ax = fig.add_subplot(gs[0, 4])
sm = ScalarMappable(cmap=cmap_map, norm=map_norm)
fig.colorbar(sm, cax=cbar_ax).set_label('Glacier fraction of streamflow (%)', fontsize=9)

fig.legend(
    handles=[
        Patch(facecolor='#e74c3c', edgecolor='#555555', label='Glacier melt fraction'),
        Patch(facecolor='#ecf0f1', edgecolor='#555555', label='Remaining streamflow'),
        Patch(facecolor=GREY, edgecolor=GREY_EDGE, label='No data'),
    ],
    loc='lower center', ncol=3, fontsize=9, bbox_to_anchor=(0.45, -0.02), framealpha=0.9)

fig.suptitle('Glacier Contribution to Streamflow — Chile (CR2MET)',
             fontsize=14, y=1.05)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/section2a_map_piecharts_3panel.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Section 2c figure: standalone megadrought-period map (JFM, 2010-2019 only)
# NOTE: this cell depends on `bounds`, `has_borders`, `chile`, `arg`,
# `cmap_map`, `GREY`, `GREY_EDGE`, `BLUE_EDGE` from the Section 2a map
# cell, which must run first. Editor red-underlines here are expected
# (static linter can't see cross-cell kernel state) and not a real error
# as long as Section 2a has already been run in this session.
# FIXED: basins sorted largest-to-smallest so smaller/nested basins draw
# on top and remain visible, same fix applied to Section 2a's map.

bounds['frac_drought_JFM'] = bounds['gauge_id'].apply(
    lambda g: period_frac(g, 'CR2MET', GAUGE_INFO_FRAC[g]['clusters'][0],
                          range(2010,2020), JFM_MONTHS)
    if g in GAUGE_INFO_FRAC else np.nan)

# Re-sort by area (bounds was already sorted this way in Section 2a, but
# re-applying here makes this cell robust even if bounds got reset/reloaded
# between cells)
bounds_2c = bounds.copy()
bounds_2c['_area_deg2'] = bounds_2c.geometry.area
bounds_2c = bounds_2c.sort_values('_area_deg2', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 14))
if has_borders:
    chile.plot(ax=ax, facecolor='#f2f2f2', edgecolor='#999999', lw=0.5, zorder=0)
    arg.plot(ax=ax, facecolor='#fafafa', edgecolor='#999999', lw=0.5, zorder=0)

vals = bounds_2c['frac_drought_JFM'].values
vmax_dr = np.nanpercentile(vals, 95)
norm_dr = Normalize(vmin=0, vmax=vmax_dr)
face = [to_hex(cmap_map(norm_dr(v))) if not np.isnan(v) else GREY for v in vals]
edge = [BLUE_EDGE if not np.isnan(v) else GREY_EDGE for v in vals]
bounds_2c.plot(ax=ax, color=face, edgecolor=edge, linewidth=0.3, zorder=2)

ax.set_xlim(-76, -60); ax.set_ylim(-56, -17); ax.set_aspect('equal')
ax.grid(True, alpha=0.15, lw=0.3)
sm = ScalarMappable(cmap=cmap_map, norm=norm_dr)
cbar = fig.colorbar(sm, ax=ax, shrink=0.6)
cbar.set_label('Glacier fraction of streamflow, JFM (%)', fontsize=9)

ax.set_title('Section 2c: Glacier Contribution During Megadrought\n'
             '(CR2MET, JFM, 2010-2019)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/section2c_megadrought_map.png', dpi=150, bbox_inches='tight')
plt.show()

# SECTION 3 -- Q3: Relative importance over time & with drought

**(Two map variants from the original notebook -- a 3-panel Annual/JFM-
drought/JFM-full map, and a standalone JFM-drought-only map -- are
omitted here since they're visually redundant with Section 2a's map
above. Can be added back on request.)**

In [ ]:
# Trend significance: is glacier fraction significantly increasing? (lag-corrected)
print('Section 3a: Significant trend in glacier fraction, 2000-2019 (lag-corrected)')
print('='*80)

frac_trend_records = []
for gauge_id, info in GAUGE_INFO_FRAC.items():
    cl0 = info['clusters'][0]
    for ds_name in datasets:
        m = get_lag_corrected_monthly(gauge_id, ds_name, cl0)
        if m is None:
            continue
        ann = m.groupby('year').agg(glac=('glac_shifted','sum'), q=('q_m3_mon','sum')).reset_index()
        ann = ann[ann['q'] > 0]
        if len(ann) < 10:
            continue
        ann['frac'] = ann['glac'] / ann['q']
        slope, intercept, r, p, se = linregress(ann['year'], ann['frac'])
        frac_trend_records.append({
            'gauge_id': gauge_id, 'cluster': cl0, 'dataset': ds_name, 'slope': slope, 'p': p,
            'sig_increase': (p < 0.05) and (slope > 0), 'sig_decrease': (p < 0.05) and (slope < 0),
        })

df_frac_trend = pd.DataFrame(frac_trend_records)
df_frac_trend.to_csv(f'{OUT_DIR}/section3a_trend_significance.csv', index=False)

print('National % basins with significant increasing fraction:')
for ds_name in datasets:
    sub = df_frac_trend[df_frac_trend['dataset'] == ds_name]
    print(f'  {ds_name:8s} n={len(sub):3d}  sig increase={sub["sig_increase"].mean()*100:5.1f}%  '
          f'sig decrease={sub["sig_decrease"].mean()*100:5.1f}%')

print('\n% significant increase by cluster (CR2MET):')
cr2_ft = df_frac_trend[df_frac_trend['dataset'] == 'CR2MET']
for cl in CLUSTERS:
    sub = cr2_ft[cr2_ft['cluster'] == cl]
    if len(sub) == 0:
        continue
    print(f'  {cl:6s} n={len(sub):3d}  {sub["sig_increase"].mean()*100:5.1f}% sig increase')

central_andes = cr2_ft[cr2_ft['cluster'].isin(['DA1','DA2','DA3'])]
print(f'\nCentral Andes (DA1+DA2+DA3) combined: {central_andes["sig_increase"].mean()*100:.1f}% '
      f'significant increase (n={len(central_andes)})')

A majority of basins nationally show a statistically significant
increasing trend in glacier fraction over 2000-2019 (56-62% depending on
climate dataset), heavily concentrated in the Central Andes (DA2: 100%,
DA3: 79%, DA1: 57%, combining to 73.3%), while OT3 and WA6 show zero
significant basins and WA3/WA4 are weak (14%). This trend result does
not depend on the ratio-of-means bug (it's computed correctly via
per-year `linregress` on the ratio series already) -- treat this
finding as solid, pending only the basin-set comparison (#4).

In [ ]:
# Section 3a figure: % basins with significant increasing fraction, by cluster
fig, ax = plt.subplots(figsize=(11, 6))

cr2_ft = df_frac_trend[df_frac_trend['dataset'] == 'CR2MET']
cl_pcts, cl_labels = [], []
for cl in CLUSTERS:
    sub = cr2_ft[cr2_ft['cluster'] == cl]
    if len(sub) == 0:
        continue
    cl_pcts.append(sub['sig_increase'].mean()*100)
    cl_labels.append(f'{cl}\n{REGION_LABELS[cl]}\n(n={len(sub)})')

colors_bar = ['#2ecc71' if p >= 50 else '#e67e22' if p >= 20 else '#e74c3c' for p in cl_pcts]
bars = ax.bar(range(len(cl_labels)), cl_pcts, color=colors_bar, edgecolor='k', lw=0.5)
for bar, p in zip(bars, cl_pcts):
    ax.text(bar.get_x()+bar.get_width()/2, p+1.5, f'{p:.0f}%', ha='center', fontsize=9, fontweight='bold')

national_pct = cr2_ft['sig_increase'].mean()*100
ax.axhline(national_pct, color='k', ls='--', lw=1.2, label=f'National (CR2MET): {national_pct:.1f}%')
ax.set_xticks(range(len(cl_labels)))
ax.set_xticklabels(cl_labels, fontsize=8)
ax.set_ylabel('% basins with significant increasing trend', fontsize=11)
ax.set_ylim(0, 110)
ax.set_title('Section 3a: Significant Increasing Trend in Glacier Fraction by Cluster\n(CR2MET, 2000-2019, p<0.05)',
             fontsize=13, fontweight='bold')
ax.legend(handles=[
    Patch(facecolor='#2ecc71', edgecolor='k', label='>=50% sig increase'),
    Patch(facecolor='#e67e22', edgecolor='k', label='20-49% sig increase'),
    Patch(facecolor='#e74c3c', edgecolor='k', label='<20% sig increase'),
    plt.Line2D([0],[0], color='k', ls='--', lw=1.2, label=f'National (CR2MET): {national_pct:.1f}%'),
], fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/section3a_sig_increase_by_cluster.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Drought amplification: basin-relative percentile method
print(f'Section 3b: Basin-relative hydrologic drought (Q<{PCTILE_THRESHOLD}th pctile), lag-corrected')
print('='*90)

annual_records_3b = []
for gauge_id, info in GAUGE_INFO_FRAC.items():
    cl0 = info['clusters'][0]
    for ds_name in datasets:
        m = get_lag_corrected_monthly(gauge_id, ds_name, cl0)
        if m is None:
            continue
        ann = m.groupby('year').agg(glac=('glac_shifted','sum'), q=('q_m3_mon','sum')).reset_index()
        ann = ann[ann['q'] > 0]
        if len(ann) < 8:
            continue
        ann['frac'] = ann['glac'] / ann['q']
        ann['gauge_id'], ann['dataset'], ann['cluster'] = gauge_id, ds_name, cl0
        annual_records_3b.append(ann)

df_annual_3b = pd.concat(annual_records_3b, ignore_index=True)
df_annual_3b['q_thresh'] = (df_annual_3b.groupby(['gauge_id','dataset'])['q']
                            .transform(lambda x: np.percentile(x, PCTILE_THRESHOLD)))
df_annual_3b['is_drought'] = df_annual_3b['q'] < df_annual_3b['q_thresh']

basin_amp = (df_annual_3b.groupby(['gauge_id','dataset','is_drought'])['frac'].mean().reset_index()
            .pivot_table(index=['gauge_id','dataset'], columns='is_drought', values='frac').reset_index())
basin_amp.columns = ['gauge_id','dataset','frac_nondrought','frac_drought']
basin_amp = basin_amp.dropna(subset=['frac_drought','frac_nondrought'])
basin_amp['amp_factor'] = basin_amp['frac_drought'] / basin_amp['frac_nondrought']
basin_amp['cluster'] = basin_amp['gauge_id'].map(lambda g: GAUGE_INFO_FRAC[g]['clusters'][0])
basin_amp.to_csv(f'{OUT_DIR}/section3b_basin_amplification.csv', index=False)

print(f'{"Cluster":6s} {"n":4s} {"Frac non-dr":11s} {"Frac drought":12s} {"Median amp":10s} {"p":8s} {"sig"}')
print('-'*70)
for cl in CLUSTERS:
    sub = basin_amp[(basin_amp['cluster']==cl) & (basin_amp['dataset']=='CR2MET') & (basin_amp['amp_factor']<10)]
    if len(sub) < 3:
        continue
    try:
        stat, p = stats.wilcoxon(sub['frac_drought'], sub['frac_nondrought'])
    except ValueError:
        p = np.nan
    print(f'{cl:6s} {len(sub):4d} {sub["frac_nondrought"].median():11.3f} {sub["frac_drought"].median():12.3f} '
          f'{sub["amp_factor"].median():10.2f}x {p:8.4f} {sig_stars(p)}')

This cell defines each basin's own drought years (below the 20th
percentile of that basin's own annual streamflow) and compares glacier
fraction during those drought years vs. all other years -- a basin-
relative definition, distinct from the calendar-period 2000-09/2010-19
comparison in Section 2c. Glacier contribution is significantly higher
during drought years in every cluster tested, 1.4-2.9x higher, strongest
in the Central Andes and weaker but still significant in wetter
clusters. This method already uses per-basin ratios correctly (no
ratio-of-means issue) -- solid finding, pending the basin-set loop (#4)
for full robustness.

In [ ]:
# Section 3b figure: non-drought vs drought fraction, by cluster
fig, ax = plt.subplots(figsize=(11, 6))

cl_nondr, cl_dr, cl_amp, cl_labels = [], [], [], []
for cl in CLUSTERS:
    sub = basin_amp[(basin_amp['cluster']==cl) & (basin_amp['dataset']=='CR2MET') & (basin_amp['amp_factor']<10)]
    if len(sub) < 3:
        continue
    cl_nondr.append(sub['frac_nondrought'].median())
    cl_dr.append(sub['frac_drought'].median())
    cl_amp.append(sub['amp_factor'].median())
    cl_labels.append(f'{cl}\n{REGION_LABELS[cl]}\n(n={len(sub)})')

x = np.arange(len(cl_labels))
width = 0.35
ax.bar(x - width/2, cl_nondr, width, label='Non-drought years', color='#2ecc71', alpha=0.85, edgecolor='k', lw=0.5)
ax.bar(x + width/2, cl_dr, width, label=f'Drought years (Q<{PCTILE_THRESHOLD}th pctile)', color='#c0392b', alpha=0.85, edgecolor='k', lw=0.5)

for i, (nd, dr, amp) in enumerate(zip(cl_nondr, cl_dr, cl_amp)):
    ax.text(i, max(nd, dr) + max(cl_dr)*0.03, f'{amp:.2f}x', ha='center', fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(cl_labels, fontsize=8)
ax.set_ylabel('Glacier fraction of streamflow', fontsize=11)
ax.set_title(f'Section 3b: Glacier Contribution During Basin-Relative Hydrologic Drought\n'
             f'(Q < {PCTILE_THRESHOLD}th percentile of each basin\'s own record) vs. non-drought years\n'
             '(CR2MET; label = median amplification factor)', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/section3b_drought_vs_nondrought_by_cluster.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Megadrought calendar comparison figure (2010-2019 vs 2000-2009 baseline)
# Same style as Section 3b, but using the calendar period definition
# (fixed 2010-2019 drought window) instead of basin-relative percentile
# drought years. JFM scope used since it captures every cluster's peak
# ablation month.

fig, ax = plt.subplots(figsize=(11, 6))

cl_base, cl_dr, cl_amp, cl_labels = [], [], [], []
for cl in CLUSTERS:
    gauges = basins_in_cluster(cl)
    base_vals, dr_vals = [], []
    for gid in gauges:
        b = period_frac(gid, 'CR2MET', cl, BASELINE, JFM_MONTHS)
        d = period_frac(gid, 'CR2MET', cl, range(2010,2020), JFM_MONTHS)
        if not np.isnan(b) and not np.isnan(d) and b > 0:
            base_vals.append(b)
            dr_vals.append(d)
    if len(base_vals) < 3:
        continue

    med_base = np.median(base_vals)
    med_dr = np.median(dr_vals)
    amp = med_dr / med_base if med_base > 0 else np.nan

    cl_base.append(med_base)
    cl_dr.append(med_dr)
    cl_amp.append(amp)
    cl_labels.append(f'{cl}\n{REGION_LABELS[cl]}\n(n={len(base_vals)})')

x = np.arange(len(cl_labels))
width = 0.35
ax.bar(x - width/2, cl_base, width, label='2000-2009 (baseline)',
       color='#2ecc71', alpha=0.85, edgecolor='k', lw=0.5)
ax.bar(x + width/2, cl_dr, width, label='2010-2019 (Megadrought)',
       color='#c0392b', alpha=0.85, edgecolor='k', lw=0.5)

for i, (base, dr, amp) in enumerate(zip(cl_base, cl_dr, cl_amp)):
    ax.text(i, max(base, dr) + max(cl_dr)*0.03, f'{amp:.2f}x',
            ha='center', fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(cl_labels, fontsize=8)
ax.set_ylabel('Glacier fraction of streamflow, JFM (%)', fontsize=11)
ax.set_title('Glacier Contribution During the Chilean Megadrought\n'
             '2000-2009 baseline vs. 2010-2019 (calendar period definition)\n'
             '(CR2MET, JFM; label = median amplification factor)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/megadrought_calendar_baseline_vs_drought_by_cluster.png',
            dpi=150, bbox_inches='tight')
plt.show()

## Section 3c: Q-decline vs. melt-change decomposition

Actual baseline and drought fractions use the mean of yearly ratios. Only the counterfactual algebra (what if only Q had changed, what if only melt had changed) uses mean annual totals, which is the correct place for a volume-based construction.

In [ ]:
decomp_simple = []

for gauge_id, info in GAUGE_INFO_FRAC.items():
    cl = info['clusters'][0]
    for ds_name in datasets:
        res = RESULTS[ds_name].get(gauge_id)
        if res is None:
            continue
        m = res['monthly'].copy()
        cy = info.get('complete_years', YEARS)
        m = m[m['year'].isin(cy)].sort_values(['year','month']).reset_index(drop=True)
        lag = DESEAS_LAG_BY_DS[ds_name].get(cl, 0)
        m['glac_shifted'] = m['glacier_runoff_m3'].shift(lag) if lag > 0 else m['glacier_runoff_m3']
        m = m.dropna(subset=['glac_shifted', 'q_m3_mon'])
        m = m[m['q_m3_mon'] > 0]

        base = m[m['year'].isin(BASELINE)]
        drgt = m[m['year'].isin(range(2010, 2020))]
        if len(base) == 0 or len(drgt) == 0:
            continue

        base_ann = base.groupby('year').agg(glac=('glac_shifted','sum'), q=('q_m3_mon','sum'))
        base_ann = base_ann[base_ann['q'] > 0]
        dr_ann = drgt.groupby('year').agg(glac=('glac_shifted','sum'), q=('q_m3_mon','sum'))
        dr_ann = dr_ann[dr_ann['q'] > 0]
        if len(base_ann) == 0 or len(dr_ann) == 0:
            continue

        frac_base = (base_ann['glac'] / base_ann['q']).mean()
        frac_dr   = (dr_ann['glac'] / dr_ann['q']).mean()
        glac_base, q_base = base_ann['glac'].mean(), base_ann['q'].mean()
        glac_dr,   q_dr   = dr_ann['glac'].mean(),   dr_ann['q'].mean()

        delta = frac_dr - frac_base
        if abs(delta) < 1e-10 or glac_base <= 0 or q_base <= 0:
            continue

        decomp_simple.append({
            'gauge_id': gauge_id, 'cluster': cl, 'dataset': ds_name,
            'delta_frac': delta,
            'q_change_pct':    (q_dr - q_base) / q_base * 100,
            'melt_change_pct': (glac_dr - glac_base) / glac_base * 100,
        })

df_decomp_simple = pd.DataFrame(decomp_simple)
df_decomp_simple.to_csv(f'{DROUGHT_DIR}/section3c_simple_changes.csv', index=False)
print(f'{len(df_decomp_simple)} basin-dataset rows computed')
print()
print('Section 3c: median Q change vs melt change (amplifying basins only), by dataset')
print('='*80)
print(f'{"Dataset":8s} {"n amplifying":13s} {"Q change med":14s} {"Melt change med":16s} {"% amplifying"}')
print('-'*80)
for ds_name in datasets:
    sub_all = df_decomp_simple[df_decomp_simple['dataset'] == ds_name]
    amp = sub_all[sub_all['delta_frac'] > 0]
    if len(sub_all) == 0:
        continue
    print(f'{ds_name:8s} {len(amp):13d} {amp["q_change_pct"].median():+13.1f}% '
          f'{amp["melt_change_pct"].median():+15.1f}% {len(amp)/len(sub_all)*100:11.1f}%')

In [ ]:
# Q change and melt change together, by cluster (amplifying basins only):
# does DA1's mechanism differ from the rest of the country?
for ds_name in datasets:
    print(f'\n{ds_name}:')
    sub_ds = df_decomp_simple[df_decomp_simple['dataset'] == ds_name]
    amp_ds = sub_ds[sub_ds['delta_frac'] > 0]
    for cl in CLUSTERS:
        sub = amp_ds[amp_ds['cluster'] == cl]
        if len(sub) < 3:
            continue
        print(f'  {cl}: Q_change median = {sub["q_change_pct"].median():+.1f}%  '
              f'melt_change median = {sub["melt_change_pct"].median():+.1f}%  (n={len(sub)})')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharey=True)

for ax, ds_name in zip(axes, datasets):
    sub_ds = df_decomp_simple[df_decomp_simple['dataset'] == ds_name]
    amp_ds = sub_ds[sub_ds['delta_frac'] > 0]

    cl_q, cl_melt, cl_labels = [], [], []
    for cl in CLUSTERS:
        sub = amp_ds[amp_ds['cluster'] == cl]
        if len(sub) < 3:
            continue
        cl_q.append(sub['q_change_pct'].median())
        cl_melt.append(sub['melt_change_pct'].median())
        cl_labels.append(f'{cl}\n(n={len(sub)})')

    x = np.arange(len(cl_labels))
    width = 0.38
    ax.bar(x - width/2, cl_q, width, label='Streamflow change', color='#e74c3c', alpha=0.9, edgecolor='k', lw=0.5)
    ax.bar(x + width/2, cl_melt, width, label='Melt change', color='#3498db', alpha=0.9, edgecolor='k', lw=0.5)
    ax.axhline(0, color='k', lw=1)
    ax.set_xticks(x)
    ax.set_xticklabels(cl_labels, fontsize=8)
    ax.set_title(ds_name, fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    if ax == axes[0]:
        ax.set_ylabel('% change, baseline -> drought', fontsize=11)
        ax.legend(fontsize=9)

fig.suptitle('Section 3c: Streamflow vs. Melt Change\n'
             'Red = Q change | Blue = melt change (amplifying basins only, medians)',
             fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(f'{DROUGHT_DIR}/section3c_simple_by_dataset.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Section 3d -- MLR explanatory power (monthly dummies, national panel)
def load_camels_mon(fname):
    df = pd.read_csv(f'{CAMELS_FULL}/{fname}')
    df = df[(df['year']>=2000)&(df['year']<=2019)].copy()
    gauge_cols = [c for c in df.columns if c not in ['date','year','month','day']]
    return df[['year','month']+gauge_cols]

precip_df = load_camels_mon('precip_cr2met_mm_mon.csv')
tmean_df  = load_camels_mon('tmean_cr2met_C_mon.csv')
pet_df    = load_camels_mon('pet_hargreaves_mm_mon.csv')

MONTH_DUMMY_COLS = [f'month_{mo}' for mo in range(2, 13)]
PREDICTORS = ['glacier_runoff_m3','precip_mm','tmean_C','pet_mm'] + MONTH_DUMMY_COLS

REG_DATA = {}
for ds_name in datasets:
    rows = []
    for gauge_id, info in GAUGE_INFO_FRAC.items():
        cl0 = info['clusters'][0]
        m = get_lag_corrected_monthly(gauge_id, ds_name, cl0)
        if m is None or len(m) == 0:
            continue
        gid = str(gauge_id)
        mc = m.rename(columns={'glac_shifted':'glacier_runoff_m3'}).copy()
        for df_src, col in [(precip_df,'precip_mm'),(tmean_df,'tmean_C'),(pet_df,'pet_mm')]:
            if gid in df_src.columns:
                clim = df_src[['year','month',gid]].rename(columns={gid:col})
                mc = mc.merge(clim, on=['year','month'], how='left')
            else:
                mc[col] = np.nan
        for mo in range(2,13):
            mc[f'month_{mo}'] = (mc['month']==mo).astype(int)
        mc['gauge_id'], mc['cluster'] = gauge_id, cl0
        rows.append(mc)
    if rows:
        REG_DATA[ds_name] = pd.concat(rows, ignore_index=True)
        print(f'{ds_name}: {len(REG_DATA[ds_name]):,} obs')

def lmg_approx_fast(df, predictors, target='q_m3_mon', n_samples=1000, seed=42):
    rng = np.random.RandomState(seed)
    sub = df.dropna(subset=predictors+[target]).copy()
    sub = sub[sub[target] > 0]
    X_full = sub[predictors].values.astype(float)
    y = sub[target].values.astype(float)
    X_c = X_full - X_full.mean(axis=0)
    y_c = y - y.mean()
    TSS = np.sum(y_c**2)
    n_pred = len(predictors)
    lmg = {p: 0.0 for p in predictors}
    for _ in range(n_samples):
        order = rng.permutation(n_pred)
        Xo = X_c[:, order]
        Q, R = np.linalg.qr(Xo)
        proj = Q.T @ y_c
        seq_r2_contrib = (proj**2) / TSS
        for idx, p_idx in enumerate(order):
            lmg[predictors[p_idx]] += seq_r2_contrib[idx]
    return {p: v/n_samples for p, v in lmg.items()}

print('\nSection 3d: National panel regression (monthly dummies), all datasets')
PANEL_RESULTS = {}
for ds_name in datasets:
    sub = REG_DATA[ds_name].dropna(subset=PREDICTORS+['q_m3_mon'])
    sub = sub[sub['q_m3_mon'] > 0]
    X, y = sub[PREDICTORS].values, sub['q_m3_mon'].values
    mod = LinearRegression().fit(X, y)
    pred = mod.predict(X)
    ss_res = np.sum((y-pred)**2)
    r2 = 1 - ss_res/np.sum((y-y.mean())**2)
    n, k = len(y), X.shape[1]
    mse = ss_res/(n-k-1)
    se_glac = np.sqrt(mse * np.linalg.inv(X.T@X).diagonal()[0])
    t_glac = mod.coef_[0]/se_glac
    p_glac = 2*stats.t.sf(abs(t_glac), df=n-k-1)
    PANEL_RESULTS[ds_name] = {'r2':r2,'n':n,'p_glacier':p_glac}
    print(f'{ds_name:8s} n={n:,}  R2={r2:.3f}  glacier p={p_glac:.4g} {sig_stars(p_glac)}')

print('\nComputing LMG relative importance (glacier runoff), fast QR method, n=1000...')
for ds_name in datasets:
    sub = REG_DATA[ds_name].dropna(subset=PREDICTORS+['q_m3_mon'])
    sub = sub[sub['q_m3_mon'] > 0]
    lmg = lmg_approx_fast(sub, PREDICTORS, n_samples=1000)
    r2 = PANEL_RESULTS[ds_name]['r2']
    lmg_glac = lmg['glacier_runoff_m3']
    print(f'  {ds_name}: LMG(glacier)={lmg_glac:.4f}  ({lmg_glac/r2*100:.1f}% of R2={r2:.3f})')
    PANEL_RESULTS[ds_name]['lmg_glacier'] = lmg_glac
    PANEL_RESULTS[ds_name]['lmg_glacier_pct'] = lmg_glac/r2*100

print('\nSection 3d complete')

In this national panel regression controlling for precipitation,
temperature, PET, and seasonality, glacier runoff is a highly
significant predictor of monthly streamflow and uniquely explains
~31-33% of the variance the model captures -- confirming glaciers are a
consistently meaningful driver of streamflow variability nationally,
not just occasionally important during drought. This result does not
depend on the ratio-of-means fix (it's a direct regression on raw
monthly data, not a period-comparison ratio).

## Peak water: glacier geometry and melt evolution

In [ ]:
# ── All-cluster Cell: Glacier geometry + melt evolution, for peak-water check ──

geometry_by_cluster = {}
for cl in CLUSTERS:
    if cl == 'WA5':
        continue
    cl_geometry = {}
    for dataset in datasets:
        suffix  = 'TC' if dataset == 'CR2MET' else dataset
        nc_file = f'{base_path}{dataset}/{cl}/run_output_2000_2019_hydro_{suffix}_{cl}.nc'
        if not os.path.exists(nc_file):
            print(f'  Missing NC: {dataset}/{cl}')
            continue
        with xr.open_dataset(nc_file) as nc:
            cl_geometry[dataset] = pd.DataFrame({
                'year':       nc['time'].values,
                'volume_km3': (nc['volume'].sum(dim='rgi_id') / 1e9).values,
                'area_km2':   (nc['area'].sum(dim='rgi_id')   / 1e6).values
            })
    if cl_geometry:
        geometry_by_cluster[cl] = cl_geometry

# --- Bring in cluster-total glacier melt volume per year, from RESULTS,
# to check alongside volume for an actual peak-water turnover signature ---
melt_by_cluster = {}
for cl in CLUSTERS:
    if cl == 'WA5':
        continue
    cl_melt = {}
    for dataset in datasets:
        res_ds = RESULTS[dataset]  # uses whichever RESULTS is currently loaded (BASIN_SET toggle)
        basin_ids = basins_in_cluster(cl)
        frames = []
        for gid in basin_ids:
            res = res_ds.get(gid)
            if res is None:
                continue
            m = res['monthly'][['year', 'glacier_runoff_m3']]
            frames.append(m)
        if not frames:
            continue
        combined = pd.concat(frames)
        ann = combined.groupby('year')['glacier_runoff_m3'].sum().reset_index()
        ann['glacier_runoff_km3'] = ann['glacier_runoff_m3'] / 1e9
        cl_melt[dataset] = ann
    if cl_melt:
        melt_by_cluster[cl] = cl_melt

# --- Plot: volume, area, and melt volume, one figure per cluster ---
for cl, cl_geometry in geometry_by_cluster.items():
    cl_melt = melt_by_cluster.get(cl, {})

    fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))

    ax = axes[0]
    for ds in datasets:
        if ds in cl_geometry:
            d = cl_geometry[ds]
            ax.plot(d['year'], d['volume_km3'], 'o-', color=colors[ds],
                    linewidth=2.5, markersize=6, label=ds, alpha=0.8)
    ax.set_ylabel('Total Volume (km$^{3}$)', fontsize=11, fontweight='bold')
    ax.set_title(f'{cl}: Volume', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    ax = axes[1]
    for ds in datasets:
        if ds in cl_geometry:
            d = cl_geometry[ds]
            ax.plot(d['year'], d['area_km2'], 'o-', color=colors[ds],
                    linewidth=2.5, markersize=6, label=ds, alpha=0.8)
    ax.set_ylabel('Total Area (km$^{2}$)', fontsize=11, fontweight='bold')
    ax.set_title(f'{cl}: Area', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    ax = axes[2]
    for ds in datasets:
        if ds in cl_melt:
            d = cl_melt[ds]
            ax.plot(d['year'], d['glacier_runoff_km3'], 'o-', color=colors[ds],
                    linewidth=2.5, markersize=6, label=ds, alpha=0.8)
    ax.set_ylabel('Glacier Runoff (km$^{3}$/yr)', fontsize=11, fontweight='bold')
    ax.set_title(f'{cl}: Melt volume — look for a peak/turnover here', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    plt.suptitle(f'{cl} Glacier Geometry + Melt Evolution (2000-2019)',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

# --- Summary table: volume/area loss + melt trend direction, all clusters ---
print('\nAll-cluster Summary:')
print(f'{"Cluster":8s} {"Dataset":8s} {"Vol loss %":11s} {"Area loss %":12s} {"Melt trend (slope, km3/yr per yr)"}')
print('-'*80)
for cl, cl_geometry in geometry_by_cluster.items():
    for ds in datasets:
        if ds not in cl_geometry:
            continue
        d = cl_geometry[ds]
        vol_pct = (d['volume_km3'].iloc[0] - d['volume_km3'].iloc[-1]) / d['volume_km3'].iloc[0] * 100
        area_pct = (d['area_km2'].iloc[0] - d['area_km2'].iloc[-1]) / d['area_km2'].iloc[0] * 100

        melt_slope = np.nan
        if cl in melt_by_cluster and ds in melt_by_cluster[cl]:
            md = melt_by_cluster[cl][ds]
            if len(md) >= 2:
                melt_slope = np.polyfit(md['year'], md['glacier_runoff_km3'], 1)[0]

        print(f'{cl:8s} {ds:8s} {vol_pct:10.1f}% {area_pct:11.1f}% {melt_slope:+.4f}')

In [ ]:
PEAK_WATER_MONTH = 3  # March (a separate variable, so the PEAK_MONTH dict from Section 2 is not overwritten)

geometry_by_cluster = {}
for cl in CLUSTERS:
    if cl == 'WA5':
        continue
    cl_geometry = {}
    for dataset in datasets:
        suffix  = 'TC' if dataset == 'CR2MET' else dataset
        nc_file = f'{base_path}{dataset}/{cl}/run_output_2000_2019_hydro_{suffix}_{cl}.nc'
        if not os.path.exists(nc_file):
            continue
        with xr.open_dataset(nc_file) as nc:
            cl_geometry[dataset] = pd.DataFrame({
                'year':       nc['time'].values,
                'volume_km3': (nc['volume'].sum(dim='rgi_id') / 1e9).values,
                'area_km2':   (nc['area'].sum(dim='rgi_id')   / 1e6).values
            })
    if cl_geometry:
        geometry_by_cluster[cl] = cl_geometry

melt_by_cluster = {}
for cl in CLUSTERS:
    if cl == 'WA5':
        continue
    cl_melt = {}
    for dataset in datasets:
        res_ds = RESULTS[dataset]
        basin_ids = basins_in_cluster(cl)
        frames = []
        for gid in basin_ids:
            res = res_ds.get(gid)
            if res is None:
                continue
            m = res['monthly']
            m_peak = m[m['month'] == PEAK_WATER_MONTH][['year', 'glacier_runoff_m3']]
            frames.append(m_peak)
        if not frames:
            continue
        combined = pd.concat(frames)
        ann = combined.groupby('year')['glacier_runoff_m3'].sum().reset_index()
        ann['glacier_runoff_km3'] = ann['glacier_runoff_m3'] / 1e9
        cl_melt[dataset] = ann
    if cl_melt:
        melt_by_cluster[cl] = cl_melt

for cl, cl_geometry in geometry_by_cluster.items():
    cl_melt = melt_by_cluster.get(cl, {})
    fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))

    for ax, key, ylabel, title in [
        (axes[0], 'volume_km3', 'Volume (km$^3$)', f'{cl}: Volume (annual)'),
        (axes[1], 'area_km2',   'Area (km$^2$)',   f'{cl}: Area (annual)'),
    ]:
        for ds in datasets:
            if ds in cl_geometry:
                d = cl_geometry[ds]
                ax.plot(d['year'], d[key], 'o-', color=colors[ds], linewidth=2.5, markersize=6, label=ds, alpha=0.8)
        ax.set_ylabel(ylabel, fontsize=11, fontweight='bold')
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.legend(fontsize=9); ax.grid(alpha=0.3)

    ax = axes[2]
    for ds in datasets:
        if ds in cl_melt:
            d = cl_melt[ds]
            ax.plot(d['year'], d['glacier_runoff_km3'], 'o-', color=colors[ds], linewidth=2.5, markersize=6, label=ds, alpha=0.8)
    ax.set_ylabel(f'Melt in month {PEAK_WATER_MONTH} (km$^3$)', fontsize=11, fontweight='bold')
    ax.set_title(f'{cl}: Peak-month melt — look for a turnover here', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    plt.suptitle(f'{cl}: Volume/Area (annual) + Peak-Month Melt, 2000-2019', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
from scipy.optimize import curve_fit

def quad(x, a, b, c):
    return a*x**2 + b*x + c

print(f'{"Cluster":8s} {"Dataset":8s} {"Raw peak yr":12s} {"Quad-fit peak yr":17s} {"Quad shape"}')
for cl, cl_melt in melt_by_cluster.items():
    for ds in datasets:
        if ds not in cl_melt:
            continue
        d = cl_melt[ds].sort_values('year')
        x, y = d['year'].values.astype(float), d['glacier_runoff_km3'].values
        raw_peak = d.loc[d['glacier_runoff_km3'].idxmax(), 'year']
        try:
            popt, _ = curve_fit(quad, x, y)
            a, b, c = popt
            if a < 0:  # downward-opening parabola = has a real interior max
                vertex_x = -b / (2*a)
                shape = 'concave (real peak)' if x.min() < vertex_x < x.max() else 'concave, peak outside range'
            else:
                vertex_x = np.nan
                shape = 'convex (no interior peak — monotonic or U-shaped)'
        except Exception:
            vertex_x, shape = np.nan, 'fit failed'
        print(f'{cl:8s} {ds:8s} {raw_peak:<12} {vertex_x:<17.1f} {shape}')

## Open checks

Short checks that still apply on the final basin set. Rerun them and decide whether they stay.

In [ ]:
# WA1 February: does ERA5 differ from CR2MET/CRU? (an anomaly seen in the old run)
for ds_name in ['CR2MET', 'ERA5', 'CRU']:
    vals = []
    for gid in basins_in_cluster('WA1'):
        frac = basin_mean_frac(gid, ds_name, 'WA1', [2])
        if not np.isnan(frac):
            vals.append(frac)
    vals = np.array(vals)
    if len(vals) == 0:
        print(f'{ds_name}: no WA1 basins with data')
        continue
    lag = DESEAS_LAG_BY_DS[ds_name].get('WA1', 'NOT FOUND')
    print(f'{ds_name}: n={len(vals)}, mean={vals.mean():.2f}%, median={np.median(vals):.2f}%, '
          f'% of basins under 0.1%: {(vals<0.1).mean()*100:.0f}%, max={vals.max():.2f}%  | WA1 lag: {lag}')

In [ ]:
# Glacier fraction by dataset: median across basins (annual, FMA, February)
rows = []
for ds_name in datasets:
    ann_vals, fma_vals, feb_vals = [], [], []
    for gid, info in GAUGE_INFO_FRAC.items():
        cl0 = info['clusters'][0]
        a   = basin_mean_frac(gid, ds_name, cl0, None)
        f   = basin_mean_frac(gid, ds_name, cl0, FMA_MONTHS)
        feb = basin_mean_frac(gid, ds_name, cl0, [2])
        if not np.isnan(a):   ann_vals.append(a)
        if not np.isnan(f):   fma_vals.append(f)
        if not np.isnan(feb): feb_vals.append(feb)
    rows.append({'dataset': ds_name, 'n': len(ann_vals),
                 'median_annual_pct': np.median(ann_vals) if ann_vals else np.nan,
                 'median_fma_pct':    np.median(fma_vals) if fma_vals else np.nan,
                 'median_feb_pct':    np.median(feb_vals) if feb_vals else np.nan})
print(pd.DataFrame(rows).round(2).to_string(index=False))

## Export for Alexis

In [ ]:
# Export for Alexis: basin boundaries reference + gauge/glacier summary CSV
export_records = []
for gauge_id, info in GAUGE_INFO_FRAC.items():
    basin_area = info['basin_area_km2']
    glacier_area = info['glacier_area_km2']
    glacier_pct = (glacier_area / basin_area * 100) if basin_area > 0 else np.nan
    export_records.append({
        'gauge_id': gauge_id, 'gauge_name': info['gauge_name'], 'cluster': info['clusters'][0],
        'cen_lat': info['cen_lat'], 'cen_lon': info.get('cen_lon', np.nan),
        'basin_area_km2': basin_area, 'glacier_area_km2': glacier_area,
        'glacier_fraction_pct': glacier_pct, 'n_glaciers': info.get('n_glaciers', len(info.get('rgi_ids', []))),
    })

df_export = pd.DataFrame(export_records).sort_values(['cluster', 'cen_lat'], ascending=[True, False])
fname = f'{OUT_DIR}/basin_gauge_glacier_summary_for_Alexis.csv'
df_export.to_csv(fname, index=False)

print(f'Saved: {fname}')
print(f'Total basins: {len(df_export)}')
print(f'Minimum glacier_fraction_pct in set: {df_export["glacier_fraction_pct"].min():.4f}%')

## SUMMARY

**Everything below is the written summary from the OLD run (143 nested basins, 137 after the high-fraction check, and the ratio-of-sums formula in Sections 2c and 3c). Every number must be rewritten from the new output before it is used in the paper or abstract.**

# SUMMARY

## Status of numbers below

- **Confirmed / solid:** MB decline significance, volume decline,
  accumulation/ablation split, lag detection, Section 3a trend
  significance, Section 3b basin-relative amplification, Section 3d
  regression -- none of these depend on the ratio-of-means bug.
- **Corrected in this notebook, pending re-run confirmation:** Section
  2c's calendar comparison and Section 3c's Q-decline/melt-change split
  now use mean-of-ratios and loop all 3 basin sets x 3 datasets (see
  those sections above) -- re-run and update the numbers below once
  confirmed.
- **Pending the duplicate-basin check (Cell 0b):** DA1/DA3's peak-month
  and JFM-calendar percentages, until confirmed basin duplication isn't
  inflating them.
- **Pending basin-set/dataset robustness check before finalizing:** the
  aridity finding below (computed once, on `all_glacierized`/CR2MET
  only, in a companion notebook -- not yet re-verified here).

---

### Q1: How are glaciers evolving 2000-2019?

Mass balance declined significantly over the full record in 7 of 9
clusters (weakest/absent in OT3 and WA5), most robustly in WA1 and the
Central Andes (DA1, DA2, DA3; ~27-36S). Glacier volume declined
significantly in EVERY cluster and every climate dataset (1.4% to 23%
loss depending on cluster) -- this result is robust across CR2MET, ERA5,
and CRU.

**Mechanism -- reframed:** across most regions (DA2, DA3, WA1, WA2, and
parts of WA3/WA4/WA6), the decline is primarily driven by **declining
snowfall (accumulation), not increasing melt (ablation)**. This does
NOT mean melt is unimportant: a glacier only loses ice when melt exceeds
that year's snow input. When snowfall declines while melt holds roughly
steady, the same melt energy that used to be absorbed by fresh snow
starts eating directly into the glacier's ice reserve instead -- so the
glacier shrinks either way, whether the proximate cause is "less snow"
or "more melt." **DA1 is the exception and the more acute case**: there,
84-99% of the MB decline is ablation-driven -- melt itself is
increasing, a genuine warming/melt-rate signal layered on top of the
ice-reserve mechanism.

**Long-term warming context (add to key findings):** independent of the
2000-2019 study period, WA1 glacier-elevation temperatures have warmed
+0.9 to +1.2C since 1960, confirmed independently across all three
long-term climate records (CR2MET back to 1960, ERA5 to 1979, CRU to
1901) -- i.e., this is not sensitive to climate-dataset choice. This
provides longer-term physical context for why MB decline should be
expected to continue and plausibly intensify, even in years without an
active megadrought signal.

### Q2: How much do glaciers contribute to streamflow?

Nationally, glacier melt supplies ~7-12% of annual streamflow (dataset-
dependent), rising in peak ablation months. **DA1 and DA2's specific
peak-month and JFM-drought percentages are UNCONFIRMED pending the
Cell 0b duplicate-basin check** -- several rows physically exceed 100%,
most likely because some CAMELS gauges along the same river (e.g. Rio
Copiapo at three separate points) share identical glacier area/melt
totals but very different streamflow, inflating any unweighted average
across "basins" in that cluster. Do not cite DA1/DA2 peak-month numbers
until this is resolved.

This analysis used lag-corrected monthly data throughout: DA2, WA2,
WA3, WA4, and WA6 showed genuine ~1-month routing delays between melt
and streamflow; WA1's apparent 5.5-month lag was confirmed to be
seasonal aliasing (true lag = 0) once deseasonalized.

### Q3: How is glacier importance changing over time and with drought?

Glacier fraction shows a significant increasing trend in a majority of
basins nationally (56-62%), concentrated almost entirely in the Central
Andes and WA1 -- this finding is solid, independent of the ratio-of-
means issue. Glacier contribution is 1.4-2.9x higher during each
basin's own drought years than non-drought years (Section 3b,
basin-relative definition) -- also solid.

**The specific Q-decline vs. melt-change split (previously reported as
~95% Q-decline / ~3% melt-change) used a formula inconsistent with the
rest of this notebook and has been corrected above (Section 3c),
looped across all 3 basin sets x 3 datasets.** Update this paragraph
with the corrected split once confirmed -- based on the same fix
applied and verified in a companion notebook, expect the corrected
Q-decline share to land closer to ~55-60%, not ~95%, meaning melt change
plays a somewhat larger (though likely still secondary) role in drought
amplification than originally reported.

### Aridity (pending basin-set/dataset verification before finalizing)

A companion analysis found glacier fraction and drought amplification
both increase with basin aridity (PET/P), with hyper-arid basins
receiving a median ~73% of February streamflow from glacier melt. This
is a striking, policy-relevant number, but **has only been computed
once, on one basin set (`all_glacierized`) and one dataset (CR2MET)** --
given the duplicate-basin issue found in this notebook, this should be
re-verified across all 3 basin sets x 3 datasets before being reported
as a confirmed finding.

### Climate dataset choice

ERA5 estimates the highest glacier melt volumes and streamflow
contribution of the three datasets; CRU the lowest. This spread is
explained by precipitation, not temperature -- the three datasets
disagree about snowfall input by nearly the same margin they disagree
about total melt (67.6 vs. 66.9 km3/yr nationally), because OGGM's
calibration forces every dataset to hit the same geodetic MB target, so
more simulated snowfall mechanically requires more simulated melt.
Despite this, all three datasets agree closely on relative glacier
volume loss and largely agree on which clusters show significant
drought-related change -- dataset choice affects absolute magnitudes
more than the overall spatial/temporal pattern.

---

## Open items before this notebook's numbers are finalized

1. Run Cell 0b's duplicate-basin check on `all_glacierized` specifically
   (where DA1/DA3's >100% values were found) and confirm whether nested/
   duplicate CAMELS gauges are the cause.
2. If duplicates are confirmed, decide (with the team) whether to
   exclude duplicate gauges, switch the default basin set to
   `grouped_main`/`leaf_only`, or both.
3. Re-run Sections 2c and 3c (already corrected + looped above) and
   update the Q2/Q3 paragraphs above with the confirmed numbers.
4. Re-verify the aridity finding across all 3 basin sets x 3 datasets in
   whichever notebook currently owns that analysis, using the same
   `find_duplicate_basins()` check before trusting cluster-level
   aggregates.
5. Consider adding the `basin_hierarchy_utils.get_basin_set()` /
   `find_duplicate_basins()` shared functions (agreed separately) to
   avoid re-solving basin-set loading and duplicate-detection
   independently in every notebook.